In [1]:
%pip install cadquery ocp-vscode numpy scipy scikit-image trimesh pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# This cel defines the five LPBF geometry generators, and applies the global filters
# ============================================================


import cadquery as cq
import random
import math
import numpy as np
import scipy.ndimage as ndi
from skimage import measure
import trimesh
import warnings
import pandas as pd
import os
from ocp_vscode import show, set_port
# Ensure this matches the port OCP is listening on (3939 is usually the default)
set_port(3939)

# ------------------------------------------------------------------------ SCRIPT 1: Thick Hollow Geometry with Features generator ------------------------------------------------------------------------
def generate_spaced_angles(n_points, min_gap_degrees=12.0):
    min_gap_rad = math.radians(min_gap_degrees)
    if n_points * min_gap_rad >= 2 * math.pi:
        raise ValueError("n_points and min_gap_degrees combination = impossible.")
        
    angles = []
    while len(angles) < n_points:
        candidate = random.uniform(0, 2 * math.pi)
        if all(min(abs(candidate - a), 2 * math.pi - abs(candidate - a)) >= min_gap_rad for a in angles):
            angles.append(candidate)
            
    return sorted(angles)

def smooth_circular_array(arr, passes):
    n = len(arr)
    smoothed = list(arr)
    for _ in range(passes):
        temp = []
        for i in range(n):
            # avg the point with its left and right neighbors
            prev_val = smoothed[i - 1]
            curr_val = smoothed[i]
            next_val = smoothed[(i + 1) % n]
            temp.append((prev_val + curr_val + next_val) / 3)
        smoothed = temp
    return smoothed

def generate_loft_path(n_sections, total_height, n_points, angles, master_radii, max_drift, use_straight_lines, thickness_offset=0.0, vertical_smoothing=0):
    section_height = total_height / n_sections
    wp = cq.Workplane("XY")
    
    cur_x, cur_y = 0, 0
    current_radii = list(master_radii)
    
    for i in range(n_sections + 1):
        layer_points = []
        for j in range(n_points):
            angle = angles[j]
            
            r = max(0.5, current_radii[j] - thickness_offset) 
            
            px = (r * math.cos(angle)) + cur_x
            py = (r * math.sin(angle)) + cur_y
            layer_points.append((px, py))
        
        wp = wp.polyline(layer_points).close()
        
        if i < n_sections:
            
            shift_x = random.uniform(-max_drift * 0.4, max_drift * 0.4)
            shift_y = random.uniform(-max_drift * 0.4, max_drift * 0.4)
            center_move_dist = math.sqrt(shift_x**2 + shift_y**2)
            
            # The budget left over for radial expansion (dr) guarantees we never exceed 45 degrees
            remaining_budget = max_drift - center_move_dist
            """" non smooth old version
            for j in range(n_points):
                dr = random.uniform(-remaining_budget, remaining_budget)
                current_radii[j] += dr
                
                # We only need to ensure the radius doesn't collapse below a physical 1mm limit
                current_radii[j] = max(1.0, current_radii[j])
            """
            raw_dr = [random.uniform(-remaining_budget, remaining_budget) for _ in range(n_points)]
            
            # USing smooth function so adjacent points arent too far apart, preventing thin walls.

            smoothed_dr = smooth_circular_array(raw_dr, passes=vertical_smoothing)
            
            for j in range(n_points):
                current_radii[j] += smoothed_dr[j]
                current_radii[j] = max(1.0, current_radii[j])

            cur_x += shift_x
            cur_y += shift_y
            wp = wp.workplane(offset=section_height)
            
    return wp.loft(ruled=use_straight_lines, combine=True)

def create_thick_hollow_geometry():
    ### Variables, also used in UI
    n_sections = random.randint(1, 4)
    total_height = random.randint(20, 100)
    wall_thickness = random.uniform(5, 10)
    min_gap_deg = 8.0 

    max_allowable_points = int(360 / min_gap_deg) - 1
    
    n_points = random.randint(8, min(20, max_allowable_points))

    base_radius_min = 20
    base_radius_max = 45

    vertical_smoothing = random.randint(0, 1)

    angles = generate_spaced_angles(n_points, min_gap_degrees=min_gap_deg)
    
    
    master_radii = [random.uniform(base_radius_min, base_radius_max) for _ in range(n_points)] 
    
    section_height = total_height / n_sections
    
    # multiply section_height by 0.5 to give splines enough room to curve and not cause more than 45deg overhangs 
    # can be altered to reach more extremes.
    max_drift = section_height * 0.5 
    use_straight_lines = random.choice([True, False])
    
    state = random.getstate()
    
    # Outer solid volume
    random.setstate(state)
    outer_solid = generate_loft_path(
        n_sections, total_height, n_points, angles, master_radii, 
        max_drift, use_straight_lines, thickness_offset=0.0, vertical_smoothing=vertical_smoothing
    )
    
    # Inner solid volume
    random.setstate(state)
    inner_solid = generate_loft_path(
        n_sections, total_height, n_points, angles, master_radii, 
        max_drift, use_straight_lines, thickness_offset=wall_thickness, vertical_smoothing=vertical_smoothing
    )
    
    thick_walled_geometry = outer_solid.cut(inner_solid)
    
    return thick_walled_geometry, inner_solid


def create_wall(shape, inner_core):
    try:
        bbox = shape.val().BoundingBox()
        spawn_z = random.uniform(bbox.zmin + 10, bbox.zmax - 5)
        
        base_cx = (bbox.xmin + bbox.xmax) / 2
        base_cy = (bbox.ymin + bbox.ymax) / 2

        thickness = random.uniform(0.5, 1.5)
        length = random.uniform(10, 30)
        outward_height = random.uniform(5, 10) 
        rotation = random.uniform(0, 360)
        rotation_rad = math.radians(rotation)
        
        n_segments = random.randint(1, 5)
        step = length / n_segments
        
        shape_params = []
        for _ in range(n_segments):
            shape_params.append((
                random.choice(["curvy", "jagged", "straight"]),
                random.uniform(-4, 4),
                random.uniform(-7, 7)
            ))

        def get_wire(z_pos, x_offset):
            path = cq.Workplane("XY").workplane(offset=z_pos).moveTo(x_offset, 0)
            curr_x = x_offset
            for mode, val1, val2 in shape_params:
                curr_x += step
                if mode == "straight":
                    path = path.lineTo(curr_x, 0)
                elif mode == "curvy":
                    path = path.spline([(curr_x - step/2, val1), (curr_x, 0)], includeCurrent=True)
                elif mode == "jagged":
                    path = path.lineTo(curr_x - step/2, val2).lineTo(curr_x, 0)
            return path.wire().offset2D(thickness / 2, kind="arc").objects[0]

        cos_a = abs(math.cos(rotation_rad))
        sin_a = abs(math.sin(rotation_rad))
        half_w = (bbox.xmax - bbox.xmin) / 2
        half_h = (bbox.ymax - bbox.ymin) / 2
        
        if half_w * sin_a > half_h * cos_a:
            approx_radius = half_h / sin_a if sin_a > 0 else half_h
        else:
            approx_radius = half_w / cos_a if cos_a > 0 else half_w

        touch_radius = approx_radius * 0.80
        
        spawn_x = base_cx + touch_radius * math.cos(rotation_rad)
        spawn_y = base_cy + touch_radius * math.sin(rotation_rad)

        wire_bottom = get_wire(0, -spawn_z) 
        wire_middle = get_wire(spawn_z, 0)

        support_section = cq.Workplane("XY")
        support_section.ctx.pendingWires.append(wire_bottom)
        support_section.ctx.pendingWires.append(wire_middle)
        support_solid = support_section.loft(combine=True)

        upper_wall = cq.Workplane("XY").add(wire_middle).toPending().extrude(outward_height)
        combined_wall_feature = support_solid.union(upper_wall).clean()

        combined_wall_feature = (
            combined_wall_feature
            .rotate((0, 0, 0), (0, 0, 1), rotation)
            .translate((spawn_x, spawn_y, 0))
        )

        joined_shape = shape.union(combined_wall_feature).clean()
        final_shape = joined_shape.cut(inner_core)

        return final_shape

    except Exception as e:
        print(f"Mid-body wall failed: {e}")
        return shape


def apply_clean_angled_cut(shape):
    try:
        bbox = shape.val().BoundingBox()
        base_w = bbox.xmax - bbox.xmin
        base_h = bbox.zmax - bbox.zmin
        center_x = (bbox.xmin + bbox.xmax) / 2
        center_y = (bbox.ymin + bbox.ymax) / 2
        
        cutter_size = base_w * 3
        cutter = cq.Workplane("XY").box(cutter_size, cutter_size, cutter_size, centered=(True, True, False))
        
        tilt_x = random.uniform(15, 35) * random.choice([-1, 1])
        tilt_y = random.uniform(15, 35) * random.choice([-1, 1])
        
        cutter = cutter.rotate((0,0,0), (1,0,0), tilt_x)
        cutter = cutter.rotate((0,0,0), (0,1,0), tilt_y)
        
        cut_depth = base_h * random.uniform(0.1, 0.2)
        target_z = bbox.zmax - cut_depth
        
        cutter = cutter.translate((center_x, center_y, target_z))
        return shape.cut(cutter)
    except Exception as e:
        print(f"Simple cut failed: {e}")
        return shape

def apply_choppy_top(shape):
    try:
        bbox = shape.val().BoundingBox()
        width = (bbox.xmax - bbox.xmin) * 4  
        top_z = bbox.zmax
        
        n_points = 7
        pts = []
        for i in range(n_points):
            x = -width/2 + (i * width/(n_points-1))
            z = random.uniform(-4, 4) 
            pts.append((x, z))
            
        cutter = (
            cq.Workplane("XZ")
            .spline(pts)
            .lineTo(width/2, 20)  
            .lineTo(-width/2, 20)
            .close()
            .extrude(width, both=True) 
        )
        
        cutter = cutter.rotateAboutCenter((0, 0, 1), random.uniform(0, 360))
        cutter = cutter.rotateAboutCenter((1, 0, 0), random.uniform(-10, 10))
        
        cut_depth = random.uniform(3, 6)
        cutter = cutter.translate((0, 0, top_z - cut_depth))
        
        return shape.cut(cutter)
    except Exception as e:
        print(f"Choppy cut failed: {e}")
        return shape

# Assembly 1
def create_shape():
    
    shape, inner_core = create_thick_hollow_geometry()

    roof_feature_functions = [apply_clean_angled_cut, apply_choppy_top]
    nroof_features = random.randint(2, 5)
    for _ in range(nroof_features):
        feature_fn = random.choice(roof_feature_functions)
        shape = feature_fn(shape)

    feature_functions = [create_wall]
    if random.random() < 0.10:
        n_features = 1
    else:
        n_features = 0
    
    for _ in range(n_features):
        feature_fn = random.choice(feature_functions)
        shape = feature_fn(shape, inner_core)

    return shape


# ------------------------------------------------------------------------ SCRIPT 2: Radial fin geometry generator ------------------------------------------------------------------------

def create_fin_architecture():
    total_target_height = random.uniform(25, 180) 
    wall_thickness = random.uniform(0.8, 4.5) 
    n_points = random.randint(5, 10) 
    n_symmetry = random.randint(3, 12) 

    z_steps = [0.0]
    chunks = [random.uniform(6.0, 15.0) for _ in range(n_points - 1)]
    total_chunks = sum(chunks)
    for c in chunks:
        z_steps.append(z_steps[-1] + (c / total_chunks) * total_target_height)

    current_r = random.uniform(10, 45)
    wp_outer = cq.Workplane("XZ").moveTo(current_r, 0.0)

    outer_track = [(current_r, 0.0)]
    use_smooth_spline = random.choice([True, False])

    # Hub
    for i in range(1, n_points):
        prev_r, prev_z = outer_track[-1]
        next_z = z_steps[i]
        dz = next_z - prev_z
        
        # 45-degree rule constraint
        max_dr = dz * 0.90  
        dr = random.uniform(-max_dr, max_dr)
        
        
        next_r = max(5.0, prev_r + dr)
        
        if use_smooth_spline and abs(dr) > 2.0:
            mid_r = prev_r + (dr / 2.0)
            mid_z = prev_z + (dz / 2.0)
            wp_outer = wp_outer.spline([(mid_r, mid_z), (next_r, next_z)], includeCurrent=True)
        else:
            wp_outer = wp_outer.lineTo(next_r, next_z)
            
        outer_track.append((next_r, next_z))

    # inner and hollow tracks
    wp_hollow = wp_outer
    inner_track = []
    for r, z in reversed(outer_track):
        inner_r = max(2.0, r - wall_thickness)
        inner_track.append((inner_r, z))

    for ir, iz in inner_track:
        wp_hollow = wp_hollow.lineTo(ir, iz)
        
    hub_hollow = wp_hollow.close().revolve(360)
    hub_outer_solid = cq.Workplane("XZ").polyline(outer_track + [(0, total_target_height), (0, 0)]).close().revolve(360)
    hub_inner_solid = cq.Workplane("XZ").polyline([(0, 0)] + inner_track[::-1] + [(0, total_target_height)]).close().revolve(360)

    max_base_r = max(r for r, z in outer_track)
       
    absolute_max_radius = 75.0 
    available_room = absolute_max_radius - max_base_r
    
    blade_length = random.uniform(5.0, max(5.0, available_room))
    blade_thick = random.uniform(0.5, 3.0) 
    total_fin_reach = max_base_r + blade_length


    max_lean = total_target_height * random.uniform(0.1, 0.4)
    lean_dir = random.choice([-1, 1])
    lean_y = max_lean * lean_dir

    crowding_factor = n_symmetry * blade_thick
    
    # If hub too crowded, ban S-Curves to prevent self-intersecting voids
    if crowding_factor > 18:
        path_style = random.choice(["linear", "bow"])
    else:
        path_style = random.choice(["linear", "bow", "s_curve"])
    

    if path_style == "linear":
        overhang_limit = total_target_height * 0.85
    elif path_style == "bow":
        overhang_limit = total_target_height * 0.40 
    else: 
        overhang_limit = total_target_height * 0.14 

    
    min_hub_circumference = 2 * math.pi * current_r
    fin_gap = min_hub_circumference / n_symmetry
    collision_limit = fin_gap * 0.40 

    max_safe_lean = min(overhang_limit, collision_limit)
    
    lean_dir = random.choice([-1, 1])
    lean_y = random.uniform(max_safe_lean * 0.2, max_safe_lean) * lean_dir

    if path_style == "linear":
        path_pts = [(0, 0), (lean_y, total_target_height)]
    elif path_style == "bow":
        path_pts = [(0, 0), (lean_y, total_target_height * 0.5), (0, total_target_height)]
    else: 
        path_pts = [(0, 0), (lean_y, total_target_height * 0.33), (-lean_y, total_target_height * 0.66), (0, total_target_height)]

    fin_path = cq.Workplane("YZ").spline(path_pts, includeCurrent=False)

    # fin setup
    fin_style = random.choice(["rectangle", "triangle", "ellipse", "polygon"])
    use_alternating = random.choice([True, False])
    secondary_reach = max_base_r + (blade_length * random.uniform(0.3, 0.7))

    def make_fin_profile(reach):
        wp = cq.Workplane("XY")
        
        anchor_depth = max_base_r 
        total_len = reach + anchor_depth
        
        if fin_style == "rectangle":
            wp = wp.moveTo(-anchor_depth, -blade_thick/2).rect(total_len, blade_thick, centered=(False, True))
        elif fin_style == "triangle":
            wp = wp.polyline([(-anchor_depth, 0), (reach, -blade_thick/2), (-anchor_depth, blade_thick/2)]).close()
        elif fin_style == "ellipse":
            wp = wp.ellipse(total_len/2, blade_thick/2).translate(((reach - anchor_depth)/2, 0))
        elif fin_style == "polygon":
            p1 = (-anchor_depth, -blade_thick / 2)
            p2 = (reach, -blade_thick / 4)
            p3 = (reach * random.uniform(0.7, 0.95), blade_thick / 2)
            p4 = (-anchor_depth, blade_thick * random.uniform(0.2, 0.7))
            wp = wp.polyline([p1, p2, p3, p4]).close()
        return wp

    # Cut top
    r_shave_top = outer_track[-1][0] + random.uniform(0.5, 6.0)
    shave_profile = [(0, 0), (total_fin_reach, 0), (r_shave_top, total_target_height), (0, total_target_height)]
    shave_cone = cq.Workplane("XZ").polyline(shave_profile).close().revolve(360)

    raw_fin_primary = make_fin_profile(total_fin_reach).sweep(fin_path, transition='round').intersect(shave_cone)
    raw_fin_secondary = make_fin_profile(secondary_reach).sweep(fin_path, transition='round').intersect(shave_cone)

    



    # Assembly
    featured_assembly = hub_outer_solid
    
    for s in range(n_symmetry):
        if random.random() < 0.15:
            continue
            
        angle = s * (360.0 / n_symmetry)
        
        current_fin = raw_fin_secondary if (use_alternating and s % 2 != 0) else raw_fin_primary
        
        rotated_fin = current_fin.rotate((0, 0, 0), (0, 0, 1), angle)
        featured_assembly = featured_assembly.union(rotated_fin)

    final_shape = featured_assembly.cut(hub_inner_solid)
    combined_part = final_shape.union(hub_hollow).clean()

    return combined_part


# ------------------------------------------------------------------------ SCRIPT 3: Thin-wall geometry generator ------------------------------------------------------------------------

def create_advanced_chaotic_heat_exchanger():
    total_length = random.uniform(40, 150)      
    base_fin_height = random.uniform(10, 50)    
    fin_thickness = random.uniform(0.4, 1.8)   
    shroud_thick = random.uniform(1, 5)    
    
    macro_layout = random.choice(["venturi", "s_curve", "tapered"])
    
    n_control_points = 6
    y_steps = [((total_length / (n_control_points - 1)) * i) for i in range(n_control_points)]
    
    
    nominal_width = random.uniform(20, 50)
    target_pitch = random.uniform(2.0, 8.0)
    n_fins = max(3, int(nominal_width / target_pitch))
    

    inner_left_pts = []
    inner_right_pts = []


    for i, y in enumerate(y_steps):
        progress = y / total_length
        
        if macro_layout == "venturi":
            width_factor = 1.0 + 0.8 * math.sin(progress * math.pi)
            current_width = nominal_width * width_factor
            x_offset = 0.0
        elif macro_layout == "s_curve":
            current_width = nominal_width * random.uniform(0.9, 1.1)
            x_offset = 22.0 * math.sin(progress * 1.5 * math.pi)
        else: # "tapered"
            width_factor = 1.4 - 0.9 * progress
            current_width = nominal_width * width_factor
            x_offset = random.uniform(-6.0, 6.0) * progress
            
        inner_left_pts.append((x_offset - current_width / 2.0, y))
        inner_right_pts.append((x_offset + current_width / 2.0, y))

    outer_left_pts = [(x - shroud_thick, y) for x, y in inner_left_pts]
    outer_right_pts = [(x + shroud_thick, y) for x, y in inner_right_pts]

    outer_shroud_solid = (
        cq.Workplane("XY")
        .moveTo(outer_left_pts[0][0], outer_left_pts[0][1])
        .spline(outer_left_pts[1:], includeCurrent=True)
        .lineTo(outer_right_pts[-1][0], outer_right_pts[-1][1])
        .spline(outer_right_pts[::-1][1:], includeCurrent=True)
        .close()
        .extrude(base_fin_height + shroud_thick + 10.0)
    )
    
    inner_tunnel_cutter = (
        cq.Workplane("XY")
        .moveTo(inner_left_pts[0][0], inner_left_pts[0][1])
        .spline(inner_left_pts[1:], includeCurrent=True)
        .lineTo(inner_right_pts[-1][0], inner_right_pts[-1][1])
        .spline(inner_right_pts[::-1][1:], includeCurrent=True)
        .close()
        .extrude(base_fin_height + 40.0) 
        .translate((0, 0, shroud_thick)) 
    )
    
    shroud = outer_shroud_solid.cut(inner_tunnel_cutter)
    assembly = shroud

    make_fins_parallel = random.random() < 0.3
    
    for f in range(n_fins):
        t = f / (n_fins - 1) if n_fins > 1 else 0.5
        
        amplitude = random.uniform(1.8, 3.8)
        phase_offset = random.uniform(0, 2 * math.pi) 
        frequency = random.choice([1.0, 1.5, 2.0])
        
        fin_left_track = []
        fin_right_track = []
        
        for i, y in enumerate(y_steps):
            xl = inner_left_pts[i][0] - 2.0
            xr = inner_right_pts[i][0] + 2.0
            
            
            x_nominal = xl + t * (xr - xl)
            
        
            if make_fins_parallel:
                x_center = x_nominal
            else:
                wave_y = (y / total_length) * (2 * math.pi) * frequency
                x_center = x_nominal + (amplitude * math.sin(wave_y + phase_offset))
            
            fin_left_track.append((x_center - fin_thickness / 2.0, y))
            fin_right_track.append((x_center + fin_thickness / 2.0, y))
            
        fin_wp = (
            cq.Workplane("XY")
            .moveTo(fin_left_track[0][0], fin_left_track[0][1])
            .spline(fin_left_track[1:], includeCurrent=True)
            .lineTo(fin_right_track[-1][0], fin_right_track[-1][1])
            .spline(fin_right_track[::-1][1:], includeCurrent=True)
            .close()
        )
        
        fin_solid = fin_wp.extrude(base_fin_height + shroud_thick + 10.0)
        assembly = assembly.union(fin_solid)

    # Pin fins
    if random.random() < 0.50: 
       
        num_cuts = random.randint(2, 6) 
        cut_width = random.uniform(1, 0.5 * total_length / num_cuts)
        
        for i in range(1, num_cuts):
            cut_y = (total_length / num_cuts) * i
            
            cutter = (
                cq.Workplane("XY")
                .workplane(offset=2.0) 
                .center(0, cut_y)
                .box(nominal_width * 3, cut_width, base_fin_height * 3, centered=(True, True, False))
            )
            assembly = assembly.cut(cutter)




    n_top_segments = random.randint(5, 8)
    top_y_steps = [((total_length / (n_top_segments - 1)) * i) for i in range(n_top_segments)]
    
    top_profile_pts = []
    for y_val in top_y_steps:
        h_val = base_fin_height * random.uniform(0.40, 1.15)
        top_profile_pts.append((y_val, h_val))
        
    top_cut_style = random.choice(["smooth_wave", "faceted_stepped"])
    
    shaper_wp = cq.Workplane("YZ").moveTo(top_profile_pts[0][0], top_profile_pts[0][1])
    
    if top_cut_style == "smooth_wave":
        shaper_wp = shaper_wp.spline(top_profile_pts[1:], includeCurrent=True)
    else:
        for pt in top_profile_pts[1:]:
            shaper_wp = shaper_wp.lineTo(pt[0], pt[1])
            
    top_shaper_tool_wire = (
        shaper_wp
        .lineTo(total_length, base_fin_height + 60.0)
        .lineTo(0, base_fin_height + 60.0)
        .close()
    )
    
    max_x = max(abs(pt[0]) for pt in outer_left_pts + outer_right_pts) + 60.0
    top_shaper_tool_solid = top_shaper_tool_wire.extrude(max_x * 2.0, both=True)
    
    final_heat_exchanger = assembly.cut(top_shaper_tool_solid).clean()
    return final_heat_exchanger



# ------------------------------------------------------------------------ SCRIPT 4: Bracket geometry generator ------------------------------------------------------------------------

def make_standard_hole(size, shape_type):
    hw = size / 2
    if shape_type == "diamond":
        th = hw * 1.15 
        return [(0, th), (hw, 0), (0, -th), (-hw, 0)]
    elif shape_type == "elongated_diamond": 
        th = hw * random.uniform(1.8, 2.5)
        return [(0, th), (hw, 0), (0, -th), (-hw, 0)]
    elif shape_type == "hex": 
        roof_h = hw * 1.2  
        wall_h = hw * 0.4  
        w = hw * 0.7       
        return [(0, roof_h), (w, wall_h), (w, -wall_h), (0, -roof_h), (-w, -wall_h), (-w, wall_h)]
    elif shape_type == "teardrop":
        tip_y = hw * math.sqrt(2)
        pts = [(0, tip_y)]
        segments = 24
        start_angle = math.pi / 4
        end_angle = -5 * math.pi / 4
        for i in range(segments + 1):
            theta = start_angle + (end_angle - start_angle) * (i / segments)
            pts.append((hw * math.cos(theta), hw * math.sin(theta)))
        return pts
    return make_standard_hole(size, "diamond")

def generate_safe_holes(x_min, x_max, max_size, min_gap=2.5):
    holes = []
    curr_x = x_min + min_gap
    while True:
        available = x_max - curr_x - min_gap
        if available < 1.5: 
            break
        
        if random.random() < 0.30:
            size = random.uniform(1.5, min(3.5, available))
        else:
            size = random.uniform(2.5, min(max_size, available))
            
        cx = curr_x + size / 2
        holes.append((cx, size))
        curr_x += size + min_gap
        
    return holes

def apply_wavy_profile(part, length, center_x, max_y, web_y):
    waves = random.randint(1, 4) 
    max_amp = (max_y - web_y) * 0.48 
    if max_amp <= 2.0: return part 
    amplitude = random.uniform(3.0, max_amp)
    
    pts = []
    segments = 60
    start_x = center_x - length / 2
    
    for i in range(segments + 1):
        t = i / segments
        pts.append((start_x + t * length, max_y/2 - math.sin(t * math.pi * waves) * amplitude))
    for i in range(segments + 1):
        t = 1 - (i / segments)
        pts.append((start_x + t * length, -max_y/2 + math.sin(t * math.pi * waves) * amplitude))

    wavy_box = cq.Workplane("XY").polyline(pts).close().extrude(2000, both=True)
    return part.intersect(wavy_box)



def make_wavy_sloped_edge(p_top, p_bot):
    dx = p_bot[0] - p_top[0]
    dz = p_bot[1] - p_top[1] 
    line_length = math.hypot(dx, dz)
    
    if line_length < 15.0:
        return [p_top, p_bot]
    
    max_waves = max(1, int(line_length / 25.0))
    waves = random.randint(1, min(4, max_waves))
    
    amp = random.uniform(0.8, 1.5) 
    
    nx = -dz / line_length
    nz = dx / line_length
    
    segments = max(40, waves * 15)
    pts = []
    
    for i in range(segments + 1):
        t = i / segments
        base_x = p_top[0] + t * dx
        base_z = p_top[1] + t * dz
        
        dampening = math.sin(t * math.pi)
        
        wave_val = math.sin(t * math.pi * waves * 2) * amp * dampening
        
        x = base_x + (nx * wave_val)
        z = base_z + (nz * wave_val)
        
        pts.append((x, z))
        
    return pts



def apply_professional_finishing(part):
    """Robust cascading finisher mapping strictly 45-degree LPBF-safe chamfers."""
    try: part = part.faces(">Z").edges().chamfer(random.uniform(0.5, 0.8))
    except: pass
    try: part = part.edges("not |X and not |Y and not |Z").chamfer(random.uniform(0.5, 0.8))
    except: pass
    try: part = part.edges("%CIRCLE").chamfer(random.uniform(0.4, 0.7))
    except: pass
    try: part = part.edges("%LINE and not <Z").chamfer(random.uniform(0.5, 0.8))
    except: pass
    try: part = part.edges("|Z").fillet(random.uniform(1.0, 2.0))
    except: pass
    return part

# Linear
def build_linear_bracket():
    height = random.uniform(40, 200) 
    boss_len = random.uniform(10, 40)
    boss_height = random.uniform(10, 20)
    
    is_l_bracket = random.choice([True, False])
    pocket_style = random.choice(["Webbed", "Hollow-Frame"])
    
    base_y = random.uniform(30, 90)
    web_y = random.uniform(3, 7)
    pocket_wall = random.uniform(4, 8)            
    
    base_thick = random.uniform(5, 15)

    if is_l_bracket:
        a1_deg = 90
        a2_deg = random.uniform(55, 80) 
        top_x_center = random.uniform(boss_len/2, boss_len * 1.2)
    else:
        a1_deg = random.uniform(55, 80)
        a2_deg = random.uniform(55, 80)
        top_x_center = random.uniform(-15, 15)

    dx1 = height / math.tan(math.radians(a1_deg))
    dx2 = height / math.tan(math.radians(a2_deg))
    
    C = (top_x_center - boss_len/2, height)
    D = (top_x_center + boss_len/2, height)
    A = (C[0] - dx1, base_thick)
    B = (D[0] + dx2, base_thick)
    
    left_tab_len = random.uniform(15, 25) if not is_l_bracket else base_thick + 5
    right_tab_len = random.uniform(15, 25)
    
    base_min_x = A[0] - left_tab_len
    base_max_x = B[0] + right_tab_len
    base_len = base_max_x - base_min_x
    base_center_x = (base_max_x + base_min_x) / 2
    
    part = cq.Workplane("XY").workplane(offset=base_thick/2).center(base_center_x, 0).box(base_len, base_y, base_thick)

    flange_y_max = base_y * random.uniform(0.6, 0.8) 

    gusset_w = random.uniform(5, 10)
    gusset_pts = [(top_x_center, height), (top_x_center + 15, base_thick), (top_x_center - 15, base_thick)]
    part = part.union(cq.Workplane("XZ").polyline(gusset_pts).close().extrude(gusset_w/2, both=True))

    pts_left = make_wavy_sloped_edge(C, A)
    pts_left.reverse() 
    pts_right = make_wavy_sloped_edge(D, B)
    
    body = cq.Workplane("XZ").polyline(pts_left + pts_right).close().extrude(flange_y_max / 2, both=True)
    boss = cq.Workplane("XZ").center(top_x_center, height).rect(boss_len, boss_height).extrude(flange_y_max / 2, both=True)
    part = part.union(body).union(boss)

    if random.choice([True, False]): part = apply_wavy_profile(part, base_len, base_center_x, base_y, web_y)


    boss_y = random.uniform(4, 8) 
    taper_z_start = base_thick + random.uniform(1, 4)
    taper_pts = [(boss_y/2, height + 20), (base_y/2, taper_z_start), (base_y/2, -20), (-base_y/2, -20), (-base_y/2, taper_z_start), (-boss_y/2, height + 20)]
    part = part.intersect(cq.Workplane("YZ").polyline(taper_pts).close().extrude(2000, both=True))

    pocket_apex_x = top_x_center
    pocket_apex_z = height - pocket_wall
    max_dx = (pocket_apex_z - (base_thick + pocket_wall)) / 1.05 
    
    p_left_x = max(A[0] + pocket_wall, pocket_apex_x - max_dx)
    p_right_x = min(B[0] - pocket_wall, pocket_apex_x + max_dx)

    if (p_right_x - p_left_x) > 5.0: 
        pocket_pts = [(p_left_x, base_thick + pocket_wall), (p_right_x, base_thick + pocket_wall), (pocket_apex_x, pocket_apex_z)]
        
        if pocket_style == "Webbed":
            part = part.cut(cq.Workplane("XZ").workplane(offset=web_y/2).polyline(pocket_pts).close().extrude(flange_y_max * 1.5))
            part = part.cut(cq.Workplane("XZ").workplane(offset=-web_y/2).polyline(pocket_pts).close().extrude(-flange_y_max * 1.5))

            num_layers = random.randint(3, 5) 
            layer_spacing = (pocket_apex_z - base_thick - pocket_wall*2) / num_layers
            max_hole_h = layer_spacing * 0.9  
            
            for i in range(num_layers):
                cz = (base_thick + pocket_wall * 1.5) + i * layer_spacing
                z_ratio = (cz - base_thick) / (pocket_apex_z - base_thick)
                layer_left = p_left_x * (1 - z_ratio) + pocket_apex_x * z_ratio
                layer_right = p_right_x * (1 - z_ratio) + pocket_apex_x * z_ratio
                
                safe_holes = generate_safe_holes(layer_left, layer_right, max_size=max_hole_h, min_gap=2.5)
                
                for cx, h_size in safe_holes:
                    shape_t = random.choice(["diamond", "teardrop", "hex", "elongated_diamond"])
                    
                    safe_size = h_size
                    if shape_t == "elongated_diamond":
                        safe_size = h_size * 0.45 
                    elif shape_t == "diamond" or shape_t == "teardrop":
                        safe_size = h_size * 0.75
                    
                    part = part.cut(
                        cq.Workplane("XZ")
                        .center(cx, cz + random.uniform(-0.5, 0.5))
                        .polyline(make_standard_hole(safe_size, shape_t))
                        .close()
                        .extrude(flange_y_max * 1.5, both=True)
                    )
        else: 
            part = part.cut(cq.Workplane("XZ").polyline(pocket_pts).close().extrude(flange_y_max * 3, both=True))

    base_holes = []
    if not is_l_bracket and left_tab_len > 12: base_holes.append((A[0] - left_tab_len / 2, 0))
    if right_tab_len > 12: base_holes.append((B[0] + right_tab_len / 2, 0))
    
    hole_r = random.uniform(2.5, 4.0)
    for h in base_holes:
        pad = cq.Workplane("XY").center(h[0], h[1]).circle(hole_r + random.uniform(3, 5)).extrude(base_thick + random.uniform(2, 4))
        part = part.union(pad)
        
    if base_holes:
        part = part.cut(cq.Workplane("XY").workplane(offset=-1).pushPoints(base_holes).circle(hole_r).extrude(base_thick + 8))

    part = part.cut(cq.Workplane("XZ").center(top_x_center, height).polyline(make_standard_hole(boss_height * 0.4, "hex")).close().extrude(flange_y_max + 10, both=True))
    part = apply_professional_finishing(part)

    return part, f"LINEAR | {'L' if is_l_bracket else 'Inline'} | {pocket_style}"


# Radial
def build_asymmetric_radial_bracket():
    hub_shape = random.choice(["cylinder", "hexagon", "square", "hollow_collar"])
    boss_r = random.uniform(10, 30)
    hub_height = random.uniform(40, 140) 
    
    base_thick = random.uniform(3, 5)
    
    layout_type = random.choice(["V-Corner", "Y-Junction", "Star-Cross", "Offset-T"])
    if layout_type == "V-Corner": angles = [0, random.uniform(60, 130)]
    elif layout_type == "Y-Junction": angles = [0, random.uniform(100, 150), random.uniform(210, 260)]
    elif layout_type == "Offset-T": angles = [0, 180, random.uniform(70, 110)]
    else: angles = [0, random.uniform(75, 105), random.uniform(165, 195), random.uniform(255, 285)]

    hub_wp = cq.Workplane("XY")
    if hub_shape == "cylinder" or hub_shape == "hollow_collar": part = hub_wp.circle(boss_r).extrude(hub_height)
    elif hub_shape == "hexagon": part = hub_wp.polygon(6, boss_r * 2).extrude(hub_height)
    else: part = hub_wp.rect(boss_r * 1.6, boss_r * 1.6).extrude(hub_height)

    cone_r_bot = boss_r * 1.5
    cone_r_top = boss_r * random.uniform(0.4, 0.8)
    cone = cq.Workplane("XY").circle(cone_r_bot).workplane(offset=hub_height + 5).circle(cone_r_top).loft()
    part = part.intersect(cone)
    
    pocket_style = random.choice(["Webbed", "Hollow-Frame"])

    for angle in angles:
        leg_height = random.uniform(hub_height * 0.4, hub_height) 
        leg_len = random.uniform(boss_r + 20, (leg_height - base_thick) / 1.3 + boss_r)
        tab_len = random.uniform(15, 50) 
        flange_y = random.uniform(10, boss_r * 1.5) 
        web_y = random.uniform(3, 7)
        pocket_wall = random.uniform(4, 8) 
        pad_width = flange_y * random.uniform(1.1, 1.6)

        start_x = -boss_r * 0.2 
        total_pad_len = (leg_len - start_x) + tab_len
        pad_center_x = start_x + (total_pad_len / 2)
        leg = cq.Workplane("XY").workplane(offset=base_thick/2).center(pad_center_x, 0).box(total_pad_len, pad_width, base_thick)

        p_top = (boss_r * 0.8, leg_height)
        p_bot = (leg_len, base_thick)
        wavy_edge = make_wavy_sloped_edge(p_top, p_bot)
        wavy_edge.reverse() 
        
        pts = [(start_x, base_thick)] + wavy_edge + [(start_x, leg_height)]
        leg = leg.union(cq.Workplane("XZ").polyline(pts).close().extrude(flange_y / 2, both=True))

        if random.choice([True, False]): leg = apply_wavy_profile(leg, total_pad_len, pad_center_x, pad_width, web_y)
        
        pocket_apex_z = leg_height - pocket_wall
        max_dx = (pocket_apex_z - (base_thick + pocket_wall)) / 1.05
        safe_inner_leg_len = min(leg_len - pocket_wall, (boss_r * 0.7) + max_dx)
        
        if safe_inner_leg_len > (boss_r * 0.7 + 5):
            pocket_pts = [(boss_r * 0.7, base_thick + pocket_wall), (safe_inner_leg_len, base_thick + pocket_wall), (boss_r * 0.7, pocket_apex_z)]
            
            if pocket_style == "Webbed":
                pocket_right = cq.Workplane("XZ").workplane(offset=web_y/2).polyline(pocket_pts).close().extrude(flange_y * 1.5)
                pocket_left = cq.Workplane("XZ").workplane(offset=-web_y/2).polyline(pocket_pts).close().extrude(-flange_y * 1.5)
                leg = leg.cut(pocket_right).cut(pocket_left)
            
                num_layers = random.randint(3, 5)
                layer_spacing = (leg_height - base_thick - pocket_wall*2) / num_layers
                max_hole_h = layer_spacing * 0.85
                
                for i in range(num_layers):
                    cz = (base_thick + pocket_wall * 1.5) + i * layer_spacing
                    z_ratio = (cz - base_thick) / (leg_height - base_thick)
                    layer_left = boss_r * 0.7
                    layer_right = safe_inner_leg_len * (1 - z_ratio) + (boss_r * 0.7) * z_ratio
                    
                    safe_holes = generate_safe_holes(layer_left, layer_right, max_size=max_hole_h, min_gap=2.5)
                    
                    for cx, h_size in safe_holes:
                        shape_t = random.choice(["diamond", "hex", "elongated_diamond"])
                        leg = leg.cut(cq.Workplane("XZ").center(cx, cz).polyline(make_standard_hole(h_size, shape_t)).close().extrude(flange_y * 1.5, both=True))
            else:
                leg = leg.cut(cq.Workplane("XZ").polyline(pocket_pts).close().extrude(flange_y * 3, both=True))

        hole_r = random.uniform(2.5, 4.5)
        pad_x = leg_len + tab_len / 2
        pad_h = base_thick + random.uniform(1.5, 3.5)
        leg = leg.union(cq.Workplane("XY").center(pad_x, 0).circle(hole_r + random.uniform(3, 6)).extrude(pad_h))
        leg = leg.cut(cq.Workplane("XY").center(pad_x, 0).circle(hole_r).extrude(pad_h * 2, both=True))
        
        part = part.union(leg.rotate((0, 0, 0), (0, 0, 1), angle))

    if hub_shape == "hollow_collar": part = part.cut(cq.Workplane("XY").circle(boss_r * 0.65).extrude(hub_height * 3, both=True))
    else: part = part.cut(cq.Workplane("XY").circle(boss_r * random.uniform(0.2, 0.4)).extrude(hub_height * 3, both=True))

    part = apply_professional_finishing(part)
    return part, f"ASYM-RADIAL | {layout_type} | {pocket_style} | Tapered Hub"


# angled extension
def build_angled_extension_bracket():
   
    base_thick = random.uniform(3, 5)
    
    arm_base_len = random.uniform(20, 80) 
    arm_width = random.uniform(10, 50)
    h_arm = random.uniform(40, 180) 
    
    lean_angle = math.radians(random.uniform(47, 80)) 
    top_len = random.uniform(10, 50) 
    pocket_wall = random.uniform(4, 8)
    web_y = random.uniform(3, 7)
    
    pocket_style = random.choice(["Webbed", "Hollow-Frame"])
    generate_serrations = random.choice([True, False])

    p1 = (-arm_base_len/2, 0)
    p2 = (arm_base_len/2, 0)
    p3 = (p2[0] + h_arm/math.tan(lean_angle), h_arm)
    p4 = (p3[0] - top_len, h_arm)

    arm = cq.Workplane("XZ").polyline([p1, p2, p3, p4]).close().extrude(arm_width/2, both=True)

    
    top_w = arm_width * random.uniform(0.3, 0.6)
    wedge_pts = [(arm_width/2, 0), (top_w/2, h_arm+10), (-top_w/2, h_arm+10), (-arm_width/2, 0)]
    wedge = cq.Workplane("YZ").polyline(wedge_pts).close().extrude(arm_base_len * 4, both=True)
    arm = arm.intersect(wedge)

    if generate_serrations:
        num_teeth = random.randint(3, 12)
        tooth_pitch = top_len / max(1, num_teeth)
        tooth_depth = random.uniform(1.5, 4.0)
        for i in range(num_teeth):
            tx = p4[0] + (i + 0.5) * tooth_pitch
            tz = h_arm
            groove_pts = [(tx - tooth_pitch/3, tz + 1), (tx + tooth_pitch/3, tz + 1), (tx, tz - tooth_depth)]
            arm = arm.cut(cq.Workplane("XZ").polyline(groove_pts).close().extrude(arm_width + 10, both=True))

    pocket_z_bottom = pocket_wall
    pocket_apex_z = h_arm - pocket_wall - (3.0 if generate_serrations else 0)
    pck_apex_x = (p3[0] + p4[0]) / 2

    max_dx = (pocket_apex_z - pocket_z_bottom) / 1.05
    wall_left_x = p1[0] + pocket_z_bottom/math.tan(lean_angle) + pocket_wall
    wall_right_x = p2[0] + pocket_z_bottom/math.tan(lean_angle) - pocket_wall

    pck_bot_left_x = max(wall_left_x, pck_apex_x - max_dx)
    pck_bot_right_x = min(wall_right_x, pck_apex_x + max_dx)

    if (pck_bot_right_x - pck_bot_left_x) > 10:
        pocket_pts = [(pck_bot_left_x, pocket_z_bottom), (pck_bot_right_x, pocket_z_bottom), (pck_apex_x, pocket_apex_z)]
        
        if pocket_style == "Webbed":
            p_right = cq.Workplane("XZ").workplane(offset=web_y/2).polyline(pocket_pts).close().extrude(arm_width)
            p_left = cq.Workplane("XZ").workplane(offset=-web_y/2).polyline(pocket_pts).close().extrude(-arm_width)
            arm = arm.cut(p_right).cut(p_left)

            num_layers = random.randint(3, 6)
            layer_spacing = (pocket_apex_z - pocket_z_bottom) / num_layers
            max_hole_h = layer_spacing * 0.85
            
            for i in range(num_layers):
                cz = pocket_z_bottom + layer_spacing * (i + 0.5)
                z_ratio = (cz - pocket_z_bottom) / (pocket_apex_z - pocket_z_bottom)
                layer_left = pck_bot_left_x * (1 - z_ratio) + pck_apex_x * z_ratio
                layer_right = pck_bot_right_x * (1 - z_ratio) + pck_apex_x * z_ratio
                
                safe_holes = generate_safe_holes(layer_left, layer_right, max_size=max_hole_h, min_gap=2.5)
                
                for cx, h_size in safe_holes:
                    shape_t = random.choice(["hex", "elongated_diamond", "teardrop"])
                    arm = arm.cut(cq.Workplane("XZ").center(cx, cz).polyline(make_standard_hole(h_size, shape_t)).close().extrude(arm_width*1.5, both=True))
        else:
            arm = arm.cut(cq.Workplane("XZ").polyline(pocket_pts).close().extrude(arm_width*3, both=True))

    if random.choice([True, False]):
        arm = apply_wavy_profile(arm, (p3[0] - p1[0]) + 20, (p1[0]+p3[0])/2, h_arm, web_y)

    root_radius = math.hypot(arm_base_len/2, arm_width/2)
    base_x = root_radius * 2 + random.uniform(30, 60)
    base_y = root_radius * 2 + random.uniform(30, 60)
    yaw_angle = random.uniform(-180, 180) 
    
    base_style = random.choice(["rect", "circle", "chamfered_poly"])
    hole_r = random.uniform(3, 5)
    pad_h = base_thick + random.uniform(2, 4)
    holes = []
    
    if base_style == "rect" or base_style == "chamfered_poly":
        if base_style == "rect":
            part = cq.Workplane("XY").rect(base_x, base_y).extrude(base_thick)
        else:
            part = cq.Workplane("XY").rect(base_x, base_y).edges("|Z").chamfer(random.uniform(5, 15)).extrude(base_thick)
            
        mx, my = base_x/2 - 12, base_y/2 - 12
        holes = [(-mx, my), (-mx, -my), (mx, my), (mx, -my)]
        if base_style == "chamfered_poly": holes = random.sample(holes, random.randint(2, 4))
        
        scallop_r_x = base_x * random.uniform(0.2, 0.4)
        scallop_r_y = base_y * random.uniform(0.2, 0.4)
        scallops = [(0, base_y/2 + scallop_r_y*0.2), (0, -base_y/2 - scallop_r_y*0.2),
                    (base_x/2 + scallop_r_x*0.2, 0), (-base_x/2 - scallop_r_x*0.2, 0)]
        for sx, sy in scallops:
            part = part.cut(cq.Workplane("XY").center(sx, sy).circle(max(scallop_r_x, scallop_r_y)).extrude(base_thick*3, both=True))

        safe_dx, safe_dy = max(0, mx - root_radius), max(0, my - root_radius)
        offset_x, offset_y = random.uniform(-safe_dx, safe_dx), random.uniform(-safe_dy, safe_dy)

    elif base_style == "circle":
        r = max(base_x, base_y) / 2
        part = cq.Workplane("XY").circle(r).extrude(base_thick)
        num_holes = random.randint(3, 6)
        hr = r - 12
        for i in range(num_holes):
            theta = (i / num_holes) * 2 * math.pi
            holes.append((hr * math.cos(theta), hr * math.sin(theta)))
        safe_r = max(0, hr - root_radius)
        rand_angle = random.uniform(0, 2 * math.pi)
        rand_dist = random.uniform(0, safe_r)
        offset_x, offset_y = rand_dist * math.cos(rand_angle), rand_dist * math.sin(rand_angle)

    part = part.union(cq.Workplane("XY").center(offset_x, offset_y).circle(root_radius*1.1).extrude(pad_h))

    if holes:
        part = part.union(cq.Workplane("XY").pushPoints(holes).circle(hole_r + random.uniform(4, 7)).extrude(pad_h))
        part = part.cut(cq.Workplane("XY").pushPoints(holes).circle(hole_r).extrude(pad_h * 3, both=True))

    arm = arm.rotate((0,0,0), (0,0,1), yaw_angle).translate((offset_x, offset_y, base_thick - 1.5))
    part = part.union(arm, clean=True)

    part = apply_professional_finishing(part)
    return part, f"ANGLED EXT | Base: {base_style} (X-Frame) | Tapered Arm"

# Master
def generate_bracket():
    """Evenly selects between Linear, Radial, and Angled Extension architectures."""
    arch = random.choice([0,1,2])
    
    if arch == 0:
        part, metadata = build_linear_bracket()
    elif arch == 1:
        part, metadata = build_asymmetric_radial_bracket()
    else:
        part, metadata = build_angled_extension_bracket()
        
    print(f"GENERATED: {metadata}")
    return part



# ------------------------------------------------------------------------ 5 TO-inspired part generation ------------------------------------------------------------------------


#-Parameters
if True:
    SEED = None

    # Domain size in mm
    SIZE_X_MM = 300.0
    SIZE_Y_MM = 300.0
    SIZE_Z_MM = 300.0

    # Grid resolution
    # Higher = smoother but slower.
    RESOLUTION = 128

    # Target volume fraction
    TARGET_VOLFRAC = 0.06

    # Random structural layout
    N_SUPPORTS = 3
    N_LOADS = 5
    N_EXTRA_BRANCHES = 4

    # Geometry
    MIN_FEATURE_SIZE_MM = 3.0
    BASE_STRUT_RADIUS_MM = 4.0
    RADIUS_RANDOMNESS_MM = 1.0
    NODE_BLOB_RADIUS_MM = 6.0

    # Smoothing
    FIELD_SMOOTHING_SIGMA = 0.8
    MESH_SMOOTHING_ITERATIONS = 15

    # LPBF printability
    BUILD_DIRECTION = np.array([0.0, 0.0, 1.0])
    OVERHANG_ANGLE_DEG = 45.0
    ALLOW_BASE_OVERHANG_HEIGHT_MM = 5.0
    USE_SELF_SUPPORT_FILTER = True

    # Generation
    MAX_GENERATION_ATTEMPTS = 20
    MAX_CAD_FACES = 25000
    VERBOSE = True

#-Helper function
def set_random_seed(seed):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)


def make_grid(nx, ny, nz, sx, sy, sz):
    x = np.linspace(-sx / 2, sx / 2, nx)
    y = np.linspace(-sy / 2, sy / 2, ny)
    z = np.linspace(0, sz, nz)

    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    dx = sx / (nx - 1)
    dy = sy / (ny - 1)
    dz = sz / (nz - 1)

    return X, Y, Z, dx, dy, dz


def volume_fraction(binary):
    return float(np.mean(binary))


def ball_structure(radius_vox):
    r = int(max(1, radius_vox))
    ax = np.arange(-r, r + 1)
    X, Y, Z = np.meshgrid(ax, ax, ax, indexing="ij")
    return (X**2 + Y**2 + Z**2) <= r**2


def disk_structure(radius_vox):
    r = int(max(1, radius_vox))
    ax = np.arange(-r, r + 1)
    X, Y = np.meshgrid(ax, ax, indexing="ij")
    return (X**2 + Y**2) <= r**2


def keep_largest_component(binary):
    labeled, n = ndi.label(binary)

    if n == 0:
        return binary

    counts = np.bincount(labeled.ravel())
    counts[0] = 0

    largest = counts.argmax()
    return labeled == largest


# ============================================================
# RANDOM SUPPORTS AND LOADS
# ============================================================

def random_point_on_bottom(sx, sy):
    margin = 0.08
    return np.array([
        random.uniform(-sx / 2 * (1 - margin), sx / 2 * (1 - margin)),
        random.uniform(-sy / 2 * (1 - margin), sy / 2 * (1 - margin)),
        0.0
    ])


def random_point_on_upper_region(sx, sy, sz):
    side = random.choice(["top", "side_x", "side_y"])

    if side == "top":
        return np.array([
            random.uniform(-sx * 0.46, sx * 0.46),
            random.uniform(-sy * 0.46, sy * 0.46),
            random.uniform(sz * 0.78, sz * 0.96)
        ])

    if side == "side_x":
        return np.array([
            random.choice([-1, 1]) * random.uniform(sx * 0.40, sx * 0.49),
            random.uniform(-sy * 0.42, sy * 0.42),
            random.uniform(sz * 0.45, sz * 0.90)
        ])

    return np.array([
        random.uniform(-sx * 0.42, sx * 0.42),
        random.choice([-1, 1]) * random.uniform(sy * 0.40, sy * 0.49),
        random.uniform(sz * 0.45, sz * 0.90)
    ])


# ============================================================
# IMPLICIT GEOMETRY FUNCTIONS
# ============================================================

def distance_to_segment_field(X, Y, Z, a, b):
    ab = b - a
    ab2 = np.dot(ab, ab) + 1e-12

    apx = X - a[0]
    apy = Y - a[1]
    apz = Z - a[2]

    t = (apx * ab[0] + apy * ab[1] + apz * ab[2]) / ab2
    t = np.clip(t, 0.0, 1.0)

    cx = a[0] + t * ab[0]
    cy = a[1] + t * ab[1]
    cz = a[2] + t * ab[2]

    return np.sqrt((X - cx)**2 + (Y - cy)**2 + (Z - cz)**2)


def add_tube_to_field(field, X, Y, Z, a, b, radius):
    d = distance_to_segment_field(X, Y, Z, a, b)
    contribution = radius - d
    np.maximum(field, contribution, out=field)


def add_blob_to_field(field, X, Y, Z, center, radius):
    d = np.sqrt(
        (X - center[0])**2 +
        (Y - center[1])**2 +
        (Z - center[2])**2
    )
    contribution = radius - d
    np.maximum(field, contribution, out=field)


def enforce_upward_self_supporting_path(points, max_angle_deg):
    """
    Makes the path grow upward and limits horizontal movement per vertical step.
    This helps create 45-degree printable branches.
    """
    angle = math.radians(max_angle_deg)
    max_horizontal_per_vertical = math.tan(angle)

    fixed = [points[0].copy()]

    for p in points[1:]:
        prev = fixed[-1].copy()
        p = p.copy()

        dz = max(p[2] - prev[2], 1e-6)

        horizontal = p[:2] - prev[:2]
        h_len = np.linalg.norm(horizontal)

        max_h = max_horizontal_per_vertical * dz

        if h_len > max_h:
            horizontal = horizontal / h_len * max_h
            p[:2] = prev[:2] + horizontal

        p[2] = max(p[2], prev[2] + 1e-3)
        fixed.append(p)

    return fixed


def make_random_polyline(start, end, sx, sy, sz, n_mid=3):
    points = [start.copy()]

    for i in range(n_mid):
        t = (i + 1) / (n_mid + 1)

        p = (1 - t) * start + t * end

        p[0] += random.uniform(-0.12 * sx, 0.12 * sx)
        p[1] += random.uniform(-0.12 * sy, 0.12 * sy)

        p[0] = np.clip(p[0], -0.46 * sx, 0.46 * sx)
        p[1] = np.clip(p[1], -0.46 * sy, 0.46 * sy)
        p[2] = np.clip(p[2], 0.02 * sz, 0.98 * sz)

        points.append(p)

    points.append(end.copy())

    points = sorted(points, key=lambda p: p[2])
    points = enforce_upward_self_supporting_path(points, OVERHANG_ANGLE_DEG)

    return points


# ============================================================
# FIELD GENERATION
# ============================================================

def create_organic_field(X, Y, Z, sx, sy, sz):
    field = np.full(X.shape, -1e6, dtype=np.float32)

    supports = [random_point_on_bottom(sx, sy) for _ in range(N_SUPPORTS)]
    loads = [random_point_on_upper_region(sx, sy, sz) for _ in range(N_LOADS)]

    # Choose one root support so everything is connected.
    root = supports[0]

    all_nodes = []

    # Foot blobs
    for s in supports:
        c = s + np.array([0.0, 0.0, NODE_BLOB_RADIUS_MM * 0.45])
        add_blob_to_field(
            field, X, Y, Z,
            c,
            NODE_BLOB_RADIUS_MM * random.uniform(0.9, 1.3)
        )
        all_nodes.append(c)

    # Connect all supports to root close to the base.
    for s in supports[1:]:
        a = root + np.array([0.0, 0.0, NODE_BLOB_RADIUS_MM * 0.35])
        b = s + np.array([0.0, 0.0, NODE_BLOB_RADIUS_MM * 0.35])
        r = BASE_STRUT_RADIUS_MM * random.uniform(0.9, 1.3)
        add_tube_to_field(field, X, Y, Z, a, b, r)

    # Load blobs
    for l in loads:
        add_blob_to_field(
            field, X, Y, Z,
            l,
            NODE_BLOB_RADIUS_MM * random.uniform(0.7, 1.1)
        )
        all_nodes.append(l)

    # Main support-to-load branches
    for l in loads:
        s = random.choice(supports)

        path = make_random_polyline(
            s + np.array([0.0, 0.0, NODE_BLOB_RADIUS_MM * 0.3]),
            l,
            sx, sy, sz,
            n_mid=random.randint(2, 4)
        )

        for a, b in zip(path[:-1], path[1:]):
            r = BASE_STRUT_RADIUS_MM + random.uniform(
                -RADIUS_RANDOMNESS_MM,
                RADIUS_RANDOMNESS_MM
            )
            r = max(r, MIN_FEATURE_SIZE_MM / 2)
            add_tube_to_field(field, X, Y, Z, a, b, r)

        for p in path:
            all_nodes.append(p)
            if random.random() < 0.6:
                add_blob_to_field(
                    field, X, Y, Z,
                    p,
                    NODE_BLOB_RADIUS_MM * random.uniform(0.35, 0.75)
                )

    # Extra organic cross branches
    for _ in range(N_EXTRA_BRANCHES):
        a = random.choice(all_nodes)
        b = random.choice(all_nodes)

        if b[2] < a[2]:
            a, b = b, a

        if abs(b[2] - a[2]) < 0.12 * sz:
            continue

        path = make_random_polyline(
            a, b,
            sx, sy, sz,
            n_mid=random.randint(1, 3)
        )

        for p0, p1 in zip(path[:-1], path[1:]):
            r = BASE_STRUT_RADIUS_MM * random.uniform(0.55, 1.05)
            r = max(r, MIN_FEATURE_SIZE_MM / 2)
            add_tube_to_field(field, X, Y, Z, p0, p1, r)

    # Add smooth organic noise
    noise = np.random.normal(0.0, 1.0, size=field.shape)
    noise = ndi.gaussian_filter(noise, sigma=random.uniform(3.0, 5.5))
    noise = noise / (np.std(noise) + 1e-12)

    field += noise.astype(np.float32) * random.uniform(0.4, 1.2)

    return field, supports, loads


def binary_from_field_with_target_volume(field, target_volfrac):
    """
    Uses an iso-level to approach the target volume fraction.

    Important:
    We do not allow too much positive thresholding, because that can cut
    the struts apart and create disconnected fragments.
    """
    q = np.quantile(field, 1.0 - target_volfrac)

    max_shrink_level = MIN_FEATURE_SIZE_MM * 0.20
    level = min(q, max_shrink_level)

    binary = field > level

    # If too little material appears, fall back to level 0.
    if volume_fraction(binary) < 0.035:
        binary = field > 0.0

    # If still too little, grow slightly.
    grow_iter = 0
    while volume_fraction(binary) < max(0.05, 0.45 * target_volfrac) and grow_iter < 3:
        binary = ndi.binary_dilation(binary, structure=ball_structure(1))
        grow_iter += 1

    return binary


# ============================================================
# LPBF SUPPORT FILTER
# ============================================================

def self_support_filter(binary, dx, dz, overhang_angle_deg):
    """
    Voxel-level approximation:
    material in each layer must be reachable from supported material below
    within a 45-degree cone.
    """
    angle = math.radians(overhang_angle_deg)
    allowed_horizontal_mm = math.tan(angle) * dz

    r_vox = max(1, int(math.ceil(allowed_horizontal_mm / dx)))
    disk = disk_structure(r_vox)

    supported = np.zeros_like(binary, dtype=bool)

    base_layers = max(1, int(math.ceil(ALLOW_BASE_OVERHANG_HEIGHT_MM / dz)))
    base_layers = min(base_layers, binary.shape[2])

    supported[:, :, :base_layers] = binary[:, :, :base_layers]

    for k in range(base_layers, binary.shape[2]):
        reachable = ndi.binary_dilation(supported[:, :, k - 1], structure=disk)
        supported[:, :, k] = binary[:, :, k] & reachable

    return supported


# ============================================================
# MESHING
# ============================================================

def binary_to_smooth_sdf(binary, sigma_vox):
    inside = ndi.distance_transform_edt(binary)
    outside = ndi.distance_transform_edt(~binary)

    sdf = inside - outside

    if sigma_vox > 0:
        sdf = ndi.gaussian_filter(sdf, sigma=sigma_vox)

    return sdf


def mesh_from_binary(binary, dx, dy, dz):
    if binary.sum() == 0:
        raise ValueError("Cannot mesh empty binary geometry.")

    sdf = binary_to_smooth_sdf(binary, FIELD_SMOOTHING_SIGMA)

    # Padding prevents open surfaces at domain boundaries.
    sdf = np.pad(sdf, pad_width=1, mode="constant", constant_values=-10)

    verts, faces, normals, values = measure.marching_cubes(
        sdf,
        level=0.0,
        spacing=(dx, dy, dz)
    )

    # Remove padding offset
    verts -= np.array([dx, dy, dz])

    # Center X and Y around zero
    verts[:, 0] -= SIZE_X_MM / 2
    verts[:, 1] -= SIZE_Y_MM / 2

    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)

    # Keep largest mesh component
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda m: m.area)

    try:
        mesh.remove_unreferenced_vertices()
        mesh.remove_duplicate_faces()
        mesh.remove_degenerate_faces()
    except Exception:
        pass

    mesh.fix_normals()

    try:
        trimesh.smoothing.filter_taubin(
            mesh,
            lamb=0.5,
            nu=-0.53,
            iterations=MESH_SMOOTHING_ITERATIONS
        )
    except Exception:
        warnings.warn("Mesh smoothing failed, continuing without extra smoothing.")

    mesh.fix_normals()

    return mesh


# ============================================================
# TO-inspired CHECKS
# ============================================================

def approximate_min_feature_check(binary, dx, min_feature_mm):
    r_vox = max(1, int(math.ceil((min_feature_mm / 2) / dx)))
    opened = ndi.binary_opening(binary, structure=ball_structure(r_vox))

    original = binary.sum()

    if original == 0:
        return 1.0

    removed_fraction = 1.0 - opened.sum() / original
    return float(removed_fraction)


def check_overhangs(mesh, overhang_angle_deg, base_ignore_height_mm):
    normals = mesh.face_normals
    centers = mesh.triangles_center
    areas = mesh.area_faces

    limit = -math.cos(math.radians(overhang_angle_deg))

    downward_bad = normals[:, 2] < limit
    not_base = centers[:, 2] > base_ignore_height_mm

    violation_mask = downward_bad & not_base

    total_area = np.sum(areas)
    violation_area = np.sum(areas[violation_mask])

    if total_area <= 0:
        violation_percent = 100.0
    else:
        violation_percent = 100.0 * violation_area / total_area

    return {
        "overhang_violation_faces": int(np.sum(violation_mask)),
        "overhang_violation_area_percent": float(violation_percent),
        "worst_normal_z": float(np.min(normals[:, 2]))
    }


def check_mesh(mesh, binary, dx):
    comps = mesh.split(only_watertight=False)

    return {
        "mesh_faces": int(len(mesh.faces)),
        "mesh_vertices": int(len(mesh.vertices)),
        "is_watertight": bool(mesh.is_watertight),
        "is_volume": bool(mesh.is_volume),
        "connected_components": int(len(comps)),
        "voxel_volume_fraction": volume_fraction(binary),
        "small_feature_fraction_estimate": approximate_min_feature_check(
            binary,
            dx,
            MIN_FEATURE_SIZE_MM
        )
    }


# ============================================================
# FALLBACK GEOMETRY
# ============================================================

def create_fallback_field(X, Y, Z, sx, sy, sz):
    """
    Guaranteed simple connected printable-ish branching structure.
    Used only if all random attempts fail.
    """
    field = np.full(X.shape, -1e6, dtype=np.float32)

    root = np.array([0.0, 0.0, 0.0])
    trunk_top = np.array([0.0, 0.0, 0.75 * sz])

    add_blob_to_field(field, X, Y, Z, root + np.array([0, 0, 6]), 10.0)
    add_tube_to_field(field, X, Y, Z, root, trunk_top, BASE_STRUT_RADIUS_MM)

    top_points = [
        np.array([0.25 * sx, 0.20 * sy, 0.95 * sz]),
        np.array([-0.25 * sx, 0.20 * sy, 0.95 * sz]),
        np.array([0.20 * sx, -0.25 * sy, 0.92 * sz]),
        np.array([-0.20 * sx, -0.25 * sy, 0.92 * sz]),
    ]

    for p in top_points:
        add_tube_to_field(field, X, Y, Z, trunk_top, p, BASE_STRUT_RADIUS_MM * 0.9)
        add_blob_to_field(field, X, Y, Z, p, NODE_BLOB_RADIUS_MM)

    return field


# ============================================================
# GENERATION PIPELINE
# ============================================================

def generate_part():
    nx = ny = nz = RESOLUTION

    X, Y, Z, dx, dy, dz = make_grid(
        nx, ny, nz,
        SIZE_X_MM, SIZE_Y_MM, SIZE_Z_MM
    )

    best = None

    for attempt in range(1, MAX_GENERATION_ATTEMPTS + 1):
        if VERBOSE:
            print(f"\nGeneration attempt {attempt}/{MAX_GENERATION_ATTEMPTS}")

        try:
            field, supports, loads = create_organic_field(
                X, Y, Z,
                SIZE_X_MM, SIZE_Y_MM, SIZE_Z_MM
            )

            binary_raw = binary_from_field_with_target_volume(
                field,
                TARGET_VOLFRAC
            )

            # Fill tiny holes and keep connected part.
            binary_raw = ndi.binary_closing(binary_raw, structure=ball_structure(1))
            binary_raw = ndi.binary_fill_holes(binary_raw)
            binary_raw = keep_largest_component(binary_raw)

            # Optional support filter.
            binary = binary_raw

            if USE_SELF_SUPPORT_FILTER:
                filtered = self_support_filter(
                    binary_raw,
                    dx=dx,
                    dz=dz,
                    overhang_angle_deg=OVERHANG_ANGLE_DEG
                )

                filtered = keep_largest_component(filtered)

                # Important robustness fix:
                # If the support filter destroys almost everything,
                # do not discard the whole attempt immediately.
                if filtered.sum() > 0.25 * binary_raw.sum() and volume_fraction(filtered) > 0.025:
                    binary = filtered
                else:
                    if VERBOSE:
                        print("Support filter was too destructive; using unfiltered connected geometry.")

            binary = keep_largest_component(binary)

            if binary.sum() == 0:
                if VERBOSE:
                    print("Empty geometry after filtering. Retrying...")
                continue

            mesh = mesh_from_binary(binary, dx, dy, dz)

            checks = check_mesh(mesh, binary, dx)
            overhangs = check_overhangs(
                mesh,
                OVERHANG_ANGLE_DEG,
                ALLOW_BASE_OVERHANG_HEIGHT_MM
            )

            if VERBOSE:
                print("Mesh checks:")
                for k, v in checks.items():
                    print(f"  {k}: {v}")

                print("Overhang checks:")
                for k, v in overhangs.items():
                    print(f"  {k}: {v}")

            # Lower score is better.
            score = (
                checks["connected_components"],
                0 if checks["is_watertight"] else 1,
                overhangs["overhang_violation_area_percent"],
                abs(checks["voxel_volume_fraction"] - TARGET_VOLFRAC)
            )

            candidate = {
                "mesh": mesh,
                "binary": binary,
                "checks": checks,
                "overhangs": overhangs,
                "score": score
            }

            if best is None or score < best["score"]:
                best = candidate

            # Accept if good enough.
            if (
                checks["connected_components"] == 1
                and checks["is_watertight"]
                and overhangs["overhang_violation_area_percent"] < 12.0
            ):
                if VERBOSE:
                    print("Accepted geometry.")
                return mesh, binary, checks, overhangs

        except Exception as e:
            if VERBOSE:
                print(f"Attempt failed with error: {e}")
            continue

    # Critical fix:
    # If all random attempts failed, create a deterministic fallback instead
    # of doing best["mesh"] when best is None.
    if best is not None:
        warnings.warn(
            "Could not satisfy all checks perfectly. Returning the best generated result."
        )
        return best["mesh"], best["binary"], best["checks"], best["overhangs"]

    warnings.warn(
        "All random attempts failed. Creating guaranteed fallback geometry."
    )

    field = create_fallback_field(X, Y, Z, SIZE_X_MM, SIZE_Y_MM, SIZE_Z_MM)
    binary = binary_from_field_with_target_volume(field, TARGET_VOLFRAC)
    binary = ndi.binary_closing(binary, structure=ball_structure(1))
    binary = ndi.binary_fill_holes(binary)
    binary = keep_largest_component(binary)

    mesh = mesh_from_binary(binary, dx, dy, dz)
    checks = check_mesh(mesh, binary, dx)
    overhangs = check_overhangs(
        mesh,
        OVERHANG_ANGLE_DEG,
        ALLOW_BASE_OVERHANG_HEIGHT_MM
    )

    return mesh, binary, checks, overhangs


# ============================================================
# MESH SIMPLIFICATION
# ============================================================

def simplify_mesh_if_needed(mesh, max_faces):
    if len(mesh.faces) <= max_faces:
        return mesh

    if VERBOSE:
        print(f"Mesh has {len(mesh.faces)} faces. Trying to simplify to {max_faces} faces...")

    simplified = None

    try:
        simplified = mesh.simplify_quadric_decimation(face_count=max_faces)
    except Exception:
        pass

    if simplified is None:
        try:
            simplified = mesh.simplify_quadratic_decimation(max_faces)
        except Exception:
            pass

    if simplified is None:
        warnings.warn(
            "Mesh simplification failed. Try lowering RESOLUTION if CadQuery is slow."
        )
        return mesh

    simplified.fix_normals()
    return simplified


# ============================================================
# CADQUERY CONVERSION
# ============================================================

def mesh_to_cadquery_solid(mesh):
    """
    Converts triangular mesh to a CadQuery object.

    First tries to sew triangular faces into a solid.
    If that fails, it returns a CadQuery compound of triangular faces
    so that you still get a visible object instead of a crash.
    """
    cq_faces = []

    verts = mesh.vertices
    faces = mesh.faces

    for tri in faces:
        p0_np = verts[tri[0]]
        p1_np = verts[tri[1]]
        p2_np = verts[tri[2]]

        # Skip degenerate triangles
        area_vec = np.cross(p1_np - p0_np, p2_np - p0_np)
        if np.linalg.norm(area_vec) < 1e-9:
            continue

        try:
            p0 = cq.Vector(float(p0_np[0]), float(p0_np[1]), float(p0_np[2]))
            p1 = cq.Vector(float(p1_np[0]), float(p1_np[1]), float(p1_np[2]))
            p2 = cq.Vector(float(p2_np[0]), float(p2_np[1]), float(p2_np[2]))

            wire = cq.Wire.makePolygon([p0, p1, p2], close=True)
            face = cq.Face.makeFromWires(wire)
            cq_faces.append(face)

        except Exception:
            continue

    if len(cq_faces) == 0:
        raise RuntimeError("CadQuery conversion failed: no valid triangular faces.")

    try:
        shell = cq.Shell.makeShell(cq_faces)
        solid = cq.Solid.makeSolid(shell)
        return cq.Workplane("XY").add(solid)

    except Exception:
        warnings.warn(
            "Could not sew mesh into a true CadQuery solid. "
            "Returning a CadQuery face compound for visualization."
        )
        compound = cq.Compound.makeCompound(cq_faces)
        return cq.Workplane("XY").add(compound)

def create_topology_optimized_geometry():
    set_random_seed(SEED)

    mesh, binary, checks, overhangs = generate_part()

    mesh = simplify_mesh_if_needed(mesh, MAX_CAD_FACES)

    if VERBOSE:
        print("\nFinal mesh summary:")
        print(f"  faces: {len(mesh.faces)}")
        print(f"  vertices: {len(mesh.vertices)}")
        print(f"  watertight: {mesh.is_watertight}")
        print(f"  volume mesh: {mesh.is_volume}")
        print(f"  connected components: {len(mesh.split(only_watertight=False))}")

        final_overhangs = check_overhangs(
            mesh,
            OVERHANG_ANGLE_DEG,
            ALLOW_BASE_OVERHANG_HEIGHT_MM
        )

        print("\nFinal overhang summary:")
        for k, v in final_overhangs.items():
            print(f"  {k}: {v}")

        print("\nFinal voxel volume fraction:")
        print(f"  {volume_fraction(binary):.3f}")

    part = mesh_to_cadquery_solid(mesh)
    return part


def generate_random_lpbf_part(weights=(1, 1, 1, 1, 1)):
   
    generators = [
        (create_shape, "lightblue"),                             # 1. Hollow Geometry
        (create_fin_architecture, "lightgreen"),                 # 2. Fin Architecture
        (create_advanced_chaotic_heat_exchanger, "yellow"),      # 3. Heat Exchanger
        (generate_bracket, "#D69B47"),                            # 4. Brackets (Light Orange hex)
        (create_topology_optimized_geometry, "red")                        #5. TO-inspired
    ]
    
    selected_item = random.choices(generators, weights=weights, k=1)[0]
    selected_function = selected_item[0]
    selected_color = selected_item[1]
    
    print(f"Executing: {selected_function.__name__}")
    
    return selected_function(), selected_color



# ------------------------------------------------------------------------ FILTER ------------------------------------------------------------------------


# Checks
def check_45_degree_overhang(
    shape,
    max_overhang_angle=45,
    max_bad_group_area_mm2=3,
    ignore_bottom_height=0.5,
    temp_filename="temp_overhang_check.stl"
):


    try:
        # 1. Export CadQuery shape temporarily as STL
        cq.exporters.export(shape, temp_filename) 

        # 2. Load STL as trimesh mesh
        mesh = trimesh.load_mesh(temp_filename)

        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))

        # Merge equal vertices before checking face connections
        mesh.merge_vertices()

        # 3. Get mesh information
        face_normals = mesh.face_normals
        face_areas = mesh.area_faces
        face_centers = mesh.triangles_center

        total_area = mesh.area
        z_min = mesh.bounds[0][2]

        if total_area <= 0:
            raise ValueError("Total mesh area is zero, so overhang area cannot be calculated.")

        bad_area = 0 
        bad_faces = 0
        bad_face_indices = []

        # 4. Find bad overhang mesh triangles
        for i, normal in enumerate(face_normals): 
            nz = normal[2] 
            face_z = face_centers[i][2]

            # Ignore bottom faces on the build plate
            if face_z <= z_min + ignore_bottom_height:
                continue

            # Only check downward-facing faces
            if nz < 0:
                surface_angle_from_horizontal = math.degrees(math.acos(abs(nz)))
                overhang_angle = 90 - surface_angle_from_horizontal 

                if overhang_angle > max_overhang_angle:
                    bad_area += face_areas[i]
                    bad_faces += 1
                    bad_face_indices.append(i)

        # 5. Group connected bad overhang faces together
        bad_face_set = set(bad_face_indices)

        bad_face_neighbours = {}

        for face_index in bad_face_indices:
            bad_face_neighbours[face_index] = []

        for face_a, face_b in mesh.face_adjacency:
            if face_a in bad_face_set and face_b in bad_face_set:
                bad_face_neighbours[face_a].append(face_b)
                bad_face_neighbours[face_b].append(face_a)

        # 6. Find connected groups of bad faces
        visited = set()
        bad_groups = []

        for face_index in bad_face_indices:
            if face_index in visited:
                continue

            group = []
            stack = [face_index]
            visited.add(face_index)

            while stack:
                current_face = stack.pop()
                group.append(current_face)

                for neighbour in bad_face_neighbours[current_face]:
                    if neighbour not in visited:
                        visited.add(neighbour)
                        stack.append(neighbour)

            bad_groups.append(group)

        # 7. Calculate largest bad group area
        largest_bad_group_area = 0
        largest_bad_group_faces = 0

        for group in bad_groups:
            group_area = sum(face_areas[face_index] for face_index in group)

            if group_area > largest_bad_group_area:
                largest_bad_group_area = group_area
                largest_bad_group_faces = len(group)

        # 8. Clean up temporary STL file
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

        # 9. Print results
        print(f"Largest bad group area: {largest_bad_group_area:.2f} mm²")

        # 10. Accept or reject based on largest grouped bad overhang area
        if largest_bad_group_area > max_bad_group_area_mm2:
            print("Part rejected: largest grouped overhang area is too large.")
            return False
        else:
            print("Part accepted: largest grouped overhang area is within allowed limits.")
            return True

    except Exception as e:
        print(f"Overhang check failed: {e}")
        return False


def check_geometry_validity(part):

    try:
 
        # 1. Keep only the largest CadQuery solid body
        solids = part.solids().vals()

        number_of_solids = len(solids)
        print(f"Number of CadQuery solid bodies before filtering: {number_of_solids}")

        if number_of_solids == 0:
            print("Part rejected: no solid bodies found.")
            return False, part

        if number_of_solids > 1:
            largest_solid = max(solids, key=lambda solid: solid.Volume())
            largest_volume = largest_solid.Volume()

            print("Smaller separate bodies were removed.")

            part = cq.Workplane("XY").add(largest_solid)
        else:
            print("Only one CadQuery solid body found.")


        # 2. Convert remaining CadQuery solid to triangle mesh
        solid = part.val()

        vertices, faces = solid.tessellate(0.1)

        mesh = trimesh.Trimesh(
            vertices=[(v.x, v.y, v.z) for v in vertices],
            faces=faces,
            process=True
        )


        # 3. Check if the mesh is a proper closed volume
        if mesh.is_volume:
            print("Part is a proper closed volume.")
        else:
            print("Part rejected: mesh is not a proper closed volume.")
            return False, part


        # 4. Check for multiple mesh components or enclosed cavities
        components = mesh.split(only_watertight=False)

        if len(components) > 1:
            print("Part rejected: part has more than 1 mesh component or an enclosed internal cavity.")
            return False, part
        else:
            print("Part has 1 mesh component.")

        return True, part

    except Exception as e:
        print(f"Additional geometry check failed: {e}")
        return False, part



# ------------------------------------------------------------------------ Global filter wrapper ------------------------------------------------------------------------

USE_GLOBAL_GEOMETRY_FILTER = True
MAX_FILTER_ATTEMPTS = 10

MAX_BAD_OVERHANG_GROUP_AREA_MM2 = 3

IGNORE_BOTTOM_HEIGHT_MM = 0.5

DEBUG_SHOW_REJECTED_OVERHANGS = False


def generate_filtered_random_lpbf_part(weights=(1, 1, 1, 1, 1)):
    """
    Generates a random LPBF part and applies the global geometry filter
    before slicing and scanning.
    """

    if not USE_GLOBAL_GEOMETRY_FILTER:
        return generate_random_lpbf_part(weights=weights)

    last_rejected_part = None
    last_rejected_color = None
    last_bad_faces_overlay = None

    for attempt in range(1, MAX_FILTER_ATTEMPTS + 1):
        print(f"\nGlobal geometry filter attempt {attempt}/{MAX_FILTER_ATTEMPTS}")

        part, display_color = generate_random_lpbf_part(weights=weights)

        # 1. General geometry validity check
        geometry_ok, part = check_geometry_validity(part)

        if not geometry_ok:
            print("Part rejected by general geometry validity check.")
            continue

        # 2. 45-degree overhang check
        overhang_ok, bad_faces_overlay = check_45_degree_overhang(
            part,
            max_overhang_angle=45,
            max_bad_group_area_mm2=MAX_BAD_OVERHANG_GROUP_AREA_MM2,
            ignore_bottom_height=IGNORE_BOTTOM_HEIGHT_MM,
            temp_filename=f"temp_overhang_check_attempt_{attempt}.stl"
        )

        if not overhang_ok:
            print("Part rejected by global overhang check.")

            last_rejected_part = part
            last_rejected_color = display_color
            last_bad_faces_overlay = bad_faces_overlay

            if DEBUG_SHOW_REJECTED_OVERHANGS and bad_faces_overlay is not None:
                show(
                    part,
                    bad_faces_overlay,
                    names=["Rejected part", "Bad overhang faces"],
                    colors=[display_color, "red"],
                    render_edges=False
                )

            continue

        print("Part accepted by global geometry filter.")
        return part, display_color

    # If no part passed after all attempts, show the last rejected part for debugging
    if last_rejected_part is not None and last_bad_faces_overlay is not None:
        show(
            last_rejected_part,
            last_bad_faces_overlay,
            names=["Last rejected part", "Bad overhang faces"],
            colors=[last_rejected_color, "red"],
            render_edges=False
        )

    raise RuntimeError("No acceptable geometry found after the global geometry filter attempts.")

In [ ]:

# ============================================================
# This cel is the batch run manager: generate + filter + slice + scan repeatedly, Place this AFTER all function definitions
# ============================================================

import time
import traceback
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import cadquery as cq
import trimesh
import random
import math


# ---------------- USER SETTINGS ----------------
N_ACCEPTED_RUNS = 1                  # <- change this to 50, 100, etc.
WEIGHTS = (0, 0, 0, 1, 0)             # <- your first test setting
MAX_TOTAL_PROCESS_ATTEMPTS = 200      # prevents infinite loop if everything keeps failing

# Fixed Desktop folders
BASE_DESKTOP = Path.home() / "Desktop"
RUN_DATA_FOLDER = BASE_DESKTOP / "LPBF_Batch_Run_Data"
TOOLPATH_FOLDER = BASE_DESKTOP / "LPBF_Dataset_CSVs"
GEOMETRY_FOLDER = BASE_DESKTOP / "LPBF_Geometries_STL"

# Batch behavior
SAVE_STL_GEOMETRIES = True            # useful later for checking the dataset visually
SAVE_COMBINED_TOOLPATH_CSV = True
SHOW_ACCEPTED_PARTS = False           # keep False for long batch runs; OCP viewer slows down batches
SKIP_EMPTY_TOOLPATHS = True
MIN_SCAN_LINES = 1

# Slicing / scanning settings
LAYER_HEIGHT_MM = 0.5
HATCH_SPACING_MM = 0.3
ROTATION_INCREMENT_DEG = 67
HATCH_POWER_W = 250
HATCH_VELOCITY_MM_S = 1000            # v in t = s / v

# Progress print settings
# 1 = print elke layer. Zet op 5 of 10 als je minder tekst wilt.
SLICE_PROGRESS_EVERY_N_LAYERS = 10
SCAN_PROGRESS_EVERY_N_LAYERS = 1
VERBOSE_PROGRESS = True

# Filter settings
MAX_FILTER_ATTEMPTS_PER_PROCESS = MAX_FILTER_ATTEMPTS
MAX_BAD_GROUP_AREA_MM2 = MAX_BAD_OVERHANG_GROUP_AREA_MM2
BOTTOM_IGNORE_HEIGHT_MM = IGNORE_BOTTOM_HEIGHT_MM

for folder in [RUN_DATA_FOLDER, TOOLPATH_FOLDER, GEOMETRY_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = RUN_DATA_FOLDER / "run_summary.csv"
FILTER_LOG_CSV = RUN_DATA_FOLDER / "filter_attempts_log.csv"
ERROR_LOG_CSV = RUN_DATA_FOLDER / "errors_log.csv"
CONFIG_JSON = RUN_DATA_FOLDER / "batch_config.json"
COMBINED_TOOLPATH_CSV = TOOLPATH_FOLDER / "all_toolpaths_combined.csv"

if SAVE_COMBINED_TOOLPATH_CSV and COMBINED_TOOLPATH_CSV.exists():
    COMBINED_TOOLPATH_CSV.unlink()

with open(CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(
        {
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "n_accepted_runs": N_ACCEPTED_RUNS,
            "weights": WEIGHTS,
            "max_total_process_attempts": MAX_TOTAL_PROCESS_ATTEMPTS,
            "layer_height_mm": LAYER_HEIGHT_MM,
            "hatch_spacing_mm": HATCH_SPACING_MM,
            "rotation_increment_deg": ROTATION_INCREMENT_DEG,
            "hatch_power_w": HATCH_POWER_W,
            "hatch_velocity_mm_s": HATCH_VELOCITY_MM_S,
            "max_filter_attempts_per_process": MAX_FILTER_ATTEMPTS_PER_PROCESS,
            "max_bad_group_area_mm2": MAX_BAD_GROUP_AREA_MM2,
            "bottom_ignore_height_mm": BOTTOM_IGNORE_HEIGHT_MM,
        },
        f,
        indent=2
    )


# ---------------- LOGGED FILTER CHECKS ----------------

def check_geometry_validity_logged(part):
    """
    Same idea as check_geometry_validity(part), but returns a statistics dictionary.
    """
    stats = {
        "geometry_validity_ok": False,
        "geometry_reject_reason": "",
        "number_of_solids_before": None,
        "number_of_solids_after": None,
        "smaller_bodies_removed": False,
        "validity_mesh_vertices": None,
        "validity_mesh_faces": None,
        "validity_is_volume": None,
        "validity_connected_components": None,
        "cadquery_volume_mm3": None,
    }

    try:
        solids = part.solids().vals()
        stats["number_of_solids_before"] = len(solids)

        if len(solids) == 0:
            stats["geometry_reject_reason"] = "no_solid_bodies_found"
            return False, part, stats

        if len(solids) > 1:
            largest_solid = max(solids, key=lambda solid: solid.Volume())
            part = cq.Workplane("XY").add(largest_solid)
            stats["smaller_bodies_removed"] = True

        stats["number_of_solids_after"] = 1

        solid = part.val()
        try:
            stats["cadquery_volume_mm3"] = float(solid.Volume())
        except Exception:
            stats["cadquery_volume_mm3"] = None

        vertices, faces = solid.tessellate(0.1)

        mesh = trimesh.Trimesh(
            vertices=[(v.x, v.y, v.z) for v in vertices],
            faces=faces,
            process=True
        )

        stats["validity_mesh_vertices"] = int(len(mesh.vertices))
        stats["validity_mesh_faces"] = int(len(mesh.faces))
        stats["validity_is_volume"] = bool(mesh.is_volume)

        if not mesh.is_volume:
            stats["geometry_reject_reason"] = "mesh_is_not_a_closed_volume"
            return False, part, stats

        components = mesh.split(only_watertight=False)
        stats["validity_connected_components"] = int(len(components))

        if len(components) > 1:
            stats["geometry_reject_reason"] = "more_than_one_mesh_component_or_internal_cavity"
            return False, part, stats

        stats["geometry_validity_ok"] = True
        stats["geometry_reject_reason"] = "accepted"
        return True, part, stats

    except Exception as e:
        stats["geometry_reject_reason"] = f"geometry_validity_error: {type(e).__name__}: {e}"
        return False, part, stats


def check_45_degree_overhang_logged(
    shape,
    max_overhang_angle=45,
    max_bad_group_area_mm2=50,
    ignore_bottom_height=5.0,
    temp_filename="temp_overhang_check.stl"
):
    """
    Same idea as check_45_degree_overhang(...), but without visual overlay by default
    and with detailed logged statistics.
    """
    stats = {
        "overhang_check_ok": False,
        "overhang_reject_reason": "",
        "mesh_total_area_mm2": None,
        "bad_overhang_area_mm2": 0.0,
        "bad_overhang_area_percent": 0.0,
        "bad_overhang_faces": 0,
        "bad_overhang_group_count": 0,
        "largest_bad_group_area_mm2": 0.0,
        "largest_bad_group_faces": 0,
        "max_detected_overhang_angle_deg": 0.0,
        "worst_normal_z": None,
    }

    temp_filename = Path(temp_filename)

    try:
        cq.exporters.export(shape, str(temp_filename))
        mesh = trimesh.load_mesh(str(temp_filename))

        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))

        mesh.merge_vertices()

        face_normals = mesh.face_normals
        face_areas = mesh.area_faces
        face_centers = mesh.triangles_center

        total_area = float(mesh.area)
        stats["mesh_total_area_mm2"] = total_area

        if total_area <= 0:
            stats["overhang_reject_reason"] = "mesh_area_zero"
            return False, stats

        z_min = float(mesh.bounds[0][2])
        stats["worst_normal_z"] = float(np.min(face_normals[:, 2]))

        bad_face_indices = []
        downward_overhang_angles = []

        for i, normal in enumerate(face_normals):
            nz = float(normal[2])
            face_z = float(face_centers[i][2])

            if face_z <= z_min + ignore_bottom_height:
                continue

            if nz < 0:
                clipped = max(-1.0, min(1.0, abs(nz)))
                surface_angle_from_horizontal = np.degrees(np.arccos(clipped))
                overhang_angle = 90.0 - surface_angle_from_horizontal
                downward_overhang_angles.append(float(overhang_angle))

                if overhang_angle > max_overhang_angle:
                    bad_face_indices.append(i)

        if downward_overhang_angles:
            stats["max_detected_overhang_angle_deg"] = float(max(downward_overhang_angles))

        bad_face_indices = list(bad_face_indices)
        bad_face_set = set(bad_face_indices)

        bad_area = float(sum(face_areas[i] for i in bad_face_indices))
        stats["bad_overhang_faces"] = int(len(bad_face_indices))
        stats["bad_overhang_area_mm2"] = bad_area
        stats["bad_overhang_area_percent"] = float(100.0 * bad_area / total_area)

        bad_face_neighbours = {face_index: [] for face_index in bad_face_indices}

        for face_a, face_b in mesh.face_adjacency:
            if face_a in bad_face_set and face_b in bad_face_set:
                bad_face_neighbours[face_a].append(face_b)
                bad_face_neighbours[face_b].append(face_a)

        visited = set()
        bad_groups = []

        for face_index in bad_face_indices:
            if face_index in visited:
                continue

            group = []
            stack = [face_index]
            visited.add(face_index)

            while stack:
                current_face = stack.pop()
                group.append(current_face)

                for neighbour in bad_face_neighbours[current_face]:
                    if neighbour not in visited:
                        visited.add(neighbour)
                        stack.append(neighbour)

            bad_groups.append(group)

        stats["bad_overhang_group_count"] = int(len(bad_groups))

        largest_bad_group_area = 0.0
        largest_bad_group_faces = 0

        for group in bad_groups:
            group_area = float(sum(face_areas[face_index] for face_index in group))
            if group_area > largest_bad_group_area:
                largest_bad_group_area = group_area
                largest_bad_group_faces = len(group)

        stats["largest_bad_group_area_mm2"] = float(largest_bad_group_area)
        stats["largest_bad_group_faces"] = int(largest_bad_group_faces)

        if largest_bad_group_area > max_bad_group_area_mm2:
            stats["overhang_reject_reason"] = "largest_bad_group_area_too_large"
            return False, stats

        stats["overhang_check_ok"] = True
        stats["overhang_reject_reason"] = "accepted"
        return True, stats

    except Exception as e:
        stats["overhang_reject_reason"] = f"overhang_check_error: {type(e).__name__}: {e}"
        return False, stats

    finally:
        try:
            if temp_filename.exists():
                temp_filename.unlink()
        except Exception:
            pass

def generate_random_lpbf_part_with_method(weights=(1, 1, 1, 1, 1)):
    generators = [
        ("hollow_geometry", create_shape, "lightblue"),
        ("fin_architecture", create_fin_architecture, "lightgreen"),
        ("heat_exchanger", create_advanced_chaotic_heat_exchanger, "yellow"),
        ("bracket", generate_bracket, "#D69B47"),
        ("topology_optimized", create_topology_optimized_geometry, "red"),
    ]

    selected_method, selected_function, selected_color = random.choices(
        generators,
        weights=weights,
        k=1
    )[0]

    print(f"Executing: {selected_method} ({selected_function.__name__})")

    part = selected_function()

    return part, selected_color, selected_method

def generate_filtered_random_lpbf_part_logged(weights, process_attempt):

    """
    Generates geometries until one passes the global filter.

    Timing definitions:
    - generation_time_s = total raw geometry-generation time in this process attempt
    - filter_time_s = total geometry-validity + overhang-check time in this process attempt
    """
    filter_attempt_rows = []
    rejected_by_geometry = 0
    rejected_by_overhang = 0
    filter_errors = 0

    total_generation_time_s = 0.0
    total_geometry_validity_time_s = 0.0
    total_overhang_check_time_s = 0.0

    for filter_attempt in range(1, MAX_FILTER_ATTEMPTS_PER_PROCESS + 1):
        row = {
            "process_attempt": process_attempt,
            "filter_attempt": filter_attempt,
            "raw_generation_time_s": None,
            "geometry_validity_time_s": None,
            "overhang_check_time_s": None,
            "filter_attempt_total_time_s": None,
            "accepted_by_filter": False,
            "reject_stage": "",
            "error_message": "",
        }

        attempt_t0 = time.perf_counter()

        try:
            print(f"  Filter attempt {filter_attempt}/{MAX_FILTER_ATTEMPTS_PER_PROCESS}", flush=True)

            # ---------------- GEOMETRY GENERATION ----------------
            print("    Generating geometry...", flush=True)

            t0 = time.perf_counter()
            part, display_color, generation_method = generate_random_lpbf_part_with_method(weights=weights)
            row["raw_generation_time_s"] = time.perf_counter() - t0

            total_generation_time_s += row["raw_generation_time_s"]
            row["generation_method"] = generation_method

            print(
                f"    Geometry generated | method = {generation_method} "
                f"| time = {row['raw_generation_time_s']:.2f} s",
                flush=True
            )

            # ---------------- GEOMETRY VALIDITY FILTER ----------------
            print("    Running geometry-validity filter...", flush=True)

            t0 = time.perf_counter()
            geometry_ok, part, geometry_stats = check_geometry_validity_logged(part)
            row["geometry_validity_time_s"] = time.perf_counter() - t0

            total_geometry_validity_time_s += row["geometry_validity_time_s"]
            row.update(geometry_stats)

            print(
                f"    Geometry-validity filter finished "
                f"| result = {'OK' if geometry_ok else 'REJECTED'} "
                f"| time = {row['geometry_validity_time_s']:.2f} s",
                flush=True
            )

            if not geometry_ok:
                rejected_by_geometry += 1
                row["reject_stage"] = "geometry_validity"
                print(f"    Reject reason: {row.get('geometry_reject_reason', '')}", flush=True)
                filter_attempt_rows.append(row)
                continue

            # ---------------- OVERHANG FILTER ----------------
            print("    Running 45-degree overhang filter...", flush=True)

            temp_stl = RUN_DATA_FOLDER / f"temp_overhang_process_{process_attempt:04d}_attempt_{filter_attempt:02d}.stl"

            t0 = time.perf_counter()
            overhang_ok, overhang_stats = check_45_degree_overhang_logged(
                part,
                max_overhang_angle=45,
                max_bad_group_area_mm2=MAX_BAD_GROUP_AREA_MM2,
                ignore_bottom_height=BOTTOM_IGNORE_HEIGHT_MM,
                temp_filename=temp_stl
            )
            row["overhang_check_time_s"] = time.perf_counter() - t0

            total_overhang_check_time_s += row["overhang_check_time_s"]
            row.update(overhang_stats)

            print(
                f"    Overhang filter finished "
                f"| result = {'OK' if overhang_ok else 'REJECTED'} "
                f"| time = {row['overhang_check_time_s']:.2f} s",
                flush=True
            )

            if not overhang_ok:
                rejected_by_overhang += 1
                row["reject_stage"] = "overhang"
                print(f"    Reject reason: {row.get('overhang_reject_reason', '')}", flush=True)
                filter_attempt_rows.append(row)
                continue

            # ---------------- ACCEPTED ----------------
            row["accepted_by_filter"] = True
            row["reject_stage"] = "accepted"
            row["filter_attempt_total_time_s"] = time.perf_counter() - attempt_t0
            filter_attempt_rows.append(row)

            total_filter_time_s = total_geometry_validity_time_s + total_overhang_check_time_s

            filter_summary = {
                "filter_success": True,
                "generation_method": row.get("generation_method", ""),
                "generation_time_s": total_generation_time_s,
                "filter_time_s": total_filter_time_s,
                "geometry_validity_total_time_s": total_geometry_validity_time_s,
                "overhang_check_total_time_s": total_overhang_check_time_s,
                "accepted_attempt_generation_time_s": row.get("raw_generation_time_s", None),
                "accepted_attempt_geometry_validity_time_s": row.get("geometry_validity_time_s", None),
                "accepted_attempt_overhang_check_time_s": row.get("overhang_check_time_s", None),
                "filter_attempts_used": filter_attempt,
                "filter_rejected_count": filter_attempt - 1,
                "filter_rejected_percentage": 100.0 * (filter_attempt - 1) / filter_attempt,
                "rejected_by_geometry_validity": rejected_by_geometry,
                "rejected_by_overhang": rejected_by_overhang,
                "filter_errors": filter_errors,
            }

            filter_summary.update(geometry_stats)
            filter_summary.update(overhang_stats)

            print(
                f"    Part accepted by filter "
                f"| total generation = {total_generation_time_s:.2f} s "
                f"| total filter = {total_filter_time_s:.2f} s",
                flush=True
            )

            return part, display_color, filter_summary, filter_attempt_rows

        except Exception as e:
            filter_errors += 1
            row["reject_stage"] = "filter_error"
            row["error_message"] = f"{type(e).__name__}: {e}"
            filter_attempt_rows.append(row)
            print(f"    Filter error: {type(e).__name__}: {e}", flush=True)
            continue

        finally:
            row["filter_attempt_total_time_s"] = time.perf_counter() - attempt_t0

    total_filter_time_s = total_geometry_validity_time_s + total_overhang_check_time_s

    filter_summary = {
        "filter_success": False,
        "generation_time_s": total_generation_time_s,
        "filter_time_s": total_filter_time_s,
        "geometry_validity_total_time_s": total_geometry_validity_time_s,
        "overhang_check_total_time_s": total_overhang_check_time_s,
        "filter_attempts_used": MAX_FILTER_ATTEMPTS_PER_PROCESS,
        "filter_rejected_count": MAX_FILTER_ATTEMPTS_PER_PROCESS,
        "filter_rejected_percentage": 100.0,
        "rejected_by_geometry_validity": rejected_by_geometry,
        "rejected_by_overhang": rejected_by_overhang,
        "filter_errors": filter_errors,
    }

    return None, None, filter_summary, filter_attempt_rows




# ---------------- SLICING AND SCANNING ----------------

def slice_part_logged(part, layer_height_mm, verbose=True, progress_every=10):
    t0 = time.perf_counter()

    slices = []
    slicer_wp = cq.Workplane("XY").add(part)

    bbox = slicer_wp.val().BoundingBox()
    z_start = bbox.zmin + 0.1
    z_end = bbox.zmax

    expected_layers = max(1, int(math.ceil((z_end - z_start) / layer_height_mm)))

    if verbose:
        print(
            f"[2/4] Slicing started "
            f"| z_start = {z_start:.2f} mm "
            f"| z_end = {z_end:.2f} mm "
            f"| expected layers = {expected_layers}",
            flush=True
        )

    current_z = z_start
    layer_index = 0

    while current_z < z_end:
        layer_t0 = time.perf_counter()

        layer_slice = slicer_wp.section(current_z)
        slices.append(layer_slice)

        layer_index += 1
        layer_time_s = time.perf_counter() - layer_t0

        if verbose and (
            layer_index == 1
            or layer_index % progress_every == 0
            or current_z + layer_height_mm >= z_end
        ):
            print(
                f"  Slice {layer_index}/{expected_layers} finished "
                f"| z = {current_z:.2f} mm "
                f"| layer time = {layer_time_s:.2f} s",
                flush=True
            )

        current_z += layer_height_mm

    slice_time_s = time.perf_counter() - t0

    if verbose:
        print(
            f"[2/4] Slicing finished "
            f"| actual slices = {len(slices)} "
            f"| total slice time = {slice_time_s:.2f} s",
            flush=True
        )

    stats = {
        "slice_time_s": slice_time_s,
        "slicing_time_s": slice_time_s,  # keep old column name too
        "slice_count": int(len(slices)),
        "bbox_xmin": float(bbox.xmin),
        "bbox_xmax": float(bbox.xmax),
        "bbox_ymin": float(bbox.ymin),
        "bbox_ymax": float(bbox.ymax),
        "bbox_zmin": float(bbox.zmin),
        "bbox_zmax": float(bbox.zmax),
        "bbox_size_x_mm": float(bbox.xmax - bbox.xmin),
        "bbox_size_y_mm": float(bbox.ymax - bbox.ymin),
        "bbox_size_z_mm": float(bbox.zmax - bbox.zmin),
    }

    return slices, stats



def scan_slices_to_dataframe_logged(
    slices,
    layer_height_mm,
    hatch_spacing_mm,
    rotation_increment_deg,
    hatch_power_w,
    hatch_velocity_mm_s,
    verbose=True,
    progress_every=1
):
    t0 = time.perf_counter()
    master_data = []

    total_layers = len(slices)

    if verbose:
        print(
            f"[3/4] Scanning started "
            f"| layers = {total_layers} "
            f"| hatch spacing = {hatch_spacing_mm} mm "
            f"| rotation increment = {rotation_increment_deg} deg",
            flush=True
        )

    for i, layer_wp in enumerate(slices):
        current_layer_num = i + 1
        current_angle = (i * rotation_increment_deg) % 360

        layer_t0 = time.perf_counter()
        lines_before_layer = len(master_data)

        if verbose and (
            current_layer_num == 1
            or current_layer_num % progress_every == 0
            or current_layer_num == total_layers
        ):
            print(
                f"  Scanning layer {current_layer_num}/{total_layers} "
                f"| angle = {current_angle:.1f} deg",
                flush=True
            )

        wire_geom = layer_wp.val()
        z_height = wire_geom.Center().z
        flat_wire = wire_geom.translate((0, 0, -z_height))
        edges = flat_wire.Edges()

        if not edges:
            if verbose and (
                current_layer_num == 1
                or current_layer_num % progress_every == 0
                or current_layer_num == total_layers
            ):
                print(f"  Scan layer {current_layer_num}/{total_layers} skipped: no edges", flush=True)
            continue

        combined_wires = cq.Wire.combine(edges)

        if not combined_wires:
            if verbose and (
                current_layer_num == 1
                or current_layer_num % progress_every == 0
                or current_layer_num == total_layers
            ):
                print(f"  Scan layer {current_layer_num}/{total_layers} skipped: no combined wires", flush=True)
            continue

        combined_wires.sort(key=lambda w: w.BoundingBox().DiagonalLength, reverse=True)
        outer_wire = combined_wires[0]
        inner_wires = combined_wires[1:]

        slice_face = cq.Face.makeFromWires(outer_wire, inner_wires)

        bb = slice_face.BoundingBox()
        center_x = (bb.xmin + bb.xmax) / 2
        center_y = (bb.ymin + bb.ymax) / 2
        diag = math.sqrt((bb.xmax - bb.xmin) ** 2 + (bb.ymax - bb.ymin) ** 2) * 1.5

        x_count = max(1, int(math.ceil(diag / hatch_spacing_mm)))

        hatch_walls = (
            cq.Workplane("XY", origin=(center_x, center_y, 0))
            .rarray(
                xSpacing=hatch_spacing_mm,
                ySpacing=diag,
                xCount=x_count,
                yCount=1,
                center=True
            )
            .rect(0.01, diag)
            .extrude(1.0)
            .rotate((center_x, center_y, 0), (center_x, center_y, 1), current_angle)
        )

        thick_slice = cq.Workplane("XY").add(slice_face).extrude(layer_height_mm)
        hatched_solid = thick_slice.intersect(hatch_walls)

        try:
            hatch_edges = hatched_solid.section().val().Edges() if hatched_solid.val() else []
        except Exception:
            hatch_edges = []

        raw_hatches = []
        normal_angle = math.radians(current_angle)
        parallel_angle = math.radians(current_angle + 90)

        for edge in hatch_edges:
            sx = edge.positionAt(0.0).x
            sy = edge.positionAt(0.0).y
            ex = edge.positionAt(1.0).x
            ey = edge.positionAt(1.0).y

            mid_x = (sx + ex) / 2
            mid_y = (sy + ey) / 2

            sweep_score = (mid_x * math.cos(normal_angle)) + (mid_y * math.sin(normal_angle))
            start_proj = (sx * math.cos(parallel_angle)) + (sy * math.sin(parallel_angle))
            end_proj = (ex * math.cos(parallel_angle)) + (ey * math.sin(parallel_angle))

            if start_proj > end_proj:
                sx, sy, ex, ey = ex, ey, sx, sy
                proj_score = end_proj
            else:
                proj_score = start_proj

            raw_hatches.append({
                "Start_X": sx,
                "Start_Y": sy,
                "End_X": ex,
                "End_Y": ey,
                "sweep_score": sweep_score,
                "proj_score": proj_score
            })

        raw_hatches.sort(key=lambda item: item["sweep_score"])

        clusters = []
        current_cluster = []
        current_score = None

        for row in raw_hatches:
            if current_score is None:
                current_cluster.append(row)
                current_score = row["sweep_score"]
            else:
                if abs(row["sweep_score"] - current_score) < (hatch_spacing_mm * 0.5):
                    current_cluster.append(row)
                else:
                    clusters.append(current_cluster)
                    current_cluster = [row]
                    current_score = row["sweep_score"]

        if current_cluster:
            clusters.append(current_cluster)

        for idx, cluster in enumerate(clusters):
            if idx % 2 != 0:
                cluster.sort(key=lambda item: item["proj_score"], reverse=True)
                for row in cluster:
                    row["Start_X"], row["Start_Y"], row["End_X"], row["End_Y"] = (
                        row["End_X"], row["End_Y"], row["Start_X"], row["Start_Y"]
                    )
            else:
                cluster.sort(key=lambda item: item["proj_score"])

            for row in cluster:
                master_data.append([
                    current_layer_num,
                    "Hatch",
                    round(row["Start_X"], 5),
                    round(row["Start_Y"], 5),
                    round(float(z_height), 5),
                    round(row["End_X"], 5),
                    round(row["End_Y"], 5),
                    round(float(z_height), 5),
                    hatch_power_w,
                    hatch_velocity_mm_s
                ])

        layer_scan_lines = len(master_data) - lines_before_layer
        layer_time_s = time.perf_counter() - layer_t0

        if verbose and (
            current_layer_num == 1
            or current_layer_num % progress_every == 0
            or current_layer_num == total_layers
        ):
            print(
                f"  Scan layer {current_layer_num}/{total_layers} finished "
                f"| angle = {current_angle:.1f} deg "
                f"| lines = {layer_scan_lines} "
                f"| layer time = {layer_time_s:.2f} s "
                f"| total lines so far = {len(master_data)}",
                flush=True
            )

    columns = [
        "Layer", "Line_Type",
        "Start_X", "Start_Y", "Start_Z",
        "End_X", "End_Y", "End_Z",
        "Power_W", "Velocity_mm_s"
    ]

    df = pd.DataFrame(master_data, columns=columns)

    if df.empty:
        total_scan_length_mm = 0.0
        estimated_laser_scan_time_s = 0.0
        average_scan_line_length_mm = 0.0
    else:
        lengths = np.sqrt(
            (df["End_X"] - df["Start_X"]) ** 2
            + (df["End_Y"] - df["Start_Y"]) ** 2
            + (df["End_Z"] - df["Start_Z"]) ** 2
        )

        df["Scan_Length_mm"] = lengths.round(5)

        total_scan_length_mm = float(lengths.sum())
        estimated_laser_scan_time_s = float((lengths / df["Velocity_mm_s"]).sum())
        average_scan_line_length_mm = float(lengths.mean())

    scan_time_s = time.perf_counter() - t0

    if verbose:
        print(
            f"[3/4] Scanning finished "
            f"| scan lines = {len(df)} "
            f"| scan time = {scan_time_s:.2f} s",
            flush=True
        )

    stats = {
        "scan_time_s": scan_time_s,
        "scanning_time_s": scan_time_s,  # keep old column name too
        "scan_line_count": int(len(df)),
        "total_scan_length_mm": total_scan_length_mm,
        "total_scan_length_m": total_scan_length_mm / 1000.0,
        "average_scan_line_length_mm": average_scan_line_length_mm,
        "estimated_laser_scan_time_s": estimated_laser_scan_time_s,
        "estimated_laser_scan_time_min": estimated_laser_scan_time_s / 60.0,
    }

    return df, stats


def save_current_logs(summary_rows, filter_attempt_rows, error_rows):
    pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
    pd.DataFrame(filter_attempt_rows).to_csv(FILTER_LOG_CSV, index=False)
    pd.DataFrame(error_rows).to_csv(ERROR_LOG_CSV, index=False)


# ---------------- MAIN BATCH LOOP ----------------

summary_rows = []
filter_attempt_rows = []
error_rows = []

accepted_runs = 0
process_attempt = 0
batch_start_time = time.perf_counter()

print(f"Starting batch generation: target = {N_ACCEPTED_RUNS} accepted runs")
print(f"Toolpath CSV folder: {TOOLPATH_FOLDER}")
print(f"Run-data folder: {RUN_DATA_FOLDER}")

while accepted_runs < N_ACCEPTED_RUNS and process_attempt < MAX_TOTAL_PROCESS_ATTEMPTS:
    process_attempt += 1
    process_t0 = time.perf_counter()

    print("\n" + "=" * 70)
    print(f"Process attempt {process_attempt} | accepted so far: {accepted_runs}/{N_ACCEPTED_RUNS}")
    print("=" * 70)

    try:
        geometry_t0 = time.perf_counter()

        part, display_color, filter_summary, current_filter_rows = generate_filtered_random_lpbf_part_logged(
            weights=WEIGHTS,
            process_attempt=process_attempt
        )

        geometry_total_time_s = time.perf_counter() - geometry_t0

        if part is None:
            for row in current_filter_rows:
                row["accepted_run_id"] = None
            filter_attempt_rows.extend(current_filter_rows)

            error_rows.append({
                "process_attempt": process_attempt,
                "stage": "filter",
                "error_message": "No geometry passed the filter in the allowed attempts.",
                "traceback": "",
                "time_s_before_error": time.perf_counter() - process_t0,
            })
            save_current_logs(summary_rows, filter_attempt_rows, error_rows)
            print("No accepted geometry from filter. Restarting with a new process attempt.")
            continue


        print(
            f"[1/4] Geometry + filter accepted "
            f"| method = {filter_summary.get('generation_method', '')} "
            f"| generation = {filter_summary.get('generation_time_s', 0):.2f} s "
            f"| filter = {filter_summary.get('filter_time_s', 0):.2f} s",
            flush=True
        )

        slices, slicing_stats = slice_part_logged(
            part,
            LAYER_HEIGHT_MM,
            verbose=VERBOSE_PROGRESS,
            progress_every=SLICE_PROGRESS_EVERY_N_LAYERS
        )

        df, scanning_stats = scan_slices_to_dataframe_logged(
            slices=slices,
            layer_height_mm=LAYER_HEIGHT_MM,
            hatch_spacing_mm=HATCH_SPACING_MM,
            rotation_increment_deg=ROTATION_INCREMENT_DEG,
            hatch_power_w=HATCH_POWER_W,
            hatch_velocity_mm_s=HATCH_VELOCITY_MM_S,
            verbose=VERBOSE_PROGRESS,
            progress_every=SCAN_PROGRESS_EVERY_N_LAYERS
        )

        print("[4/4] Saving CSV/STL outputs and logs...", flush=True)

        if SKIP_EMPTY_TOOLPATHS and len(df) < MIN_SCAN_LINES:
            for row in current_filter_rows:
                row["accepted_run_id"] = None
            filter_attempt_rows.extend(current_filter_rows)

            error_rows.append({
                "process_attempt": process_attempt,
                "stage": "scanning",
                "error_message": f"Toolpath had only {len(df)} scan lines, so it was skipped.",
                "traceback": "",
                "time_s_before_error": time.perf_counter() - process_t0,
            })
            save_current_logs(summary_rows, filter_attempt_rows, error_rows)
            print("Empty or too-small toolpath. Restarting with a new process attempt.")
            continue

        run_id = accepted_runs + 1

        for row in current_filter_rows:
            row["accepted_run_id"] = run_id
        filter_attempt_rows.extend(current_filter_rows)

        df.insert(0, "Run_ID", run_id)
        df.insert(1, "Process_Attempt", process_attempt)

        toolpath_csv_path = TOOLPATH_FOLDER / f"toolpath_run_{run_id:03d}.csv"
        df.to_csv(toolpath_csv_path, index=False)

        if SAVE_COMBINED_TOOLPATH_CSV:
            df.to_csv(
                COMBINED_TOOLPATH_CSV,
                mode="a",
                header=not COMBINED_TOOLPATH_CSV.exists(),
                index=False
            )

        geometry_stl_path = ""
        if SAVE_STL_GEOMETRIES:
            geometry_stl_path = GEOMETRY_FOLDER / f"geometry_run_{run_id:03d}.stl"
            try:
                cq.exporters.export(part, str(geometry_stl_path))
            except Exception as stl_error:
                geometry_stl_path = f"STL export failed: {type(stl_error).__name__}: {stl_error}"

        if SHOW_ACCEPTED_PARTS:
            try:
                show(part, names=[f"Accepted_Run_{run_id:03d}"], colors=[display_color], render_edges=False)
            except Exception as show_error:
                print(f"OCP show failed but run is still saved: {show_error}")

        accepted_runs = run_id
        total_run_time_s = time.perf_counter() - process_t0

        summary_row = {
            "run_id": run_id,
            "process_attempt": process_attempt,
            "status": "accepted",

            # Main timing columns
            "generation_time_s": filter_summary.get("generation_time_s", None),
            "filter_time_s": filter_summary.get("filter_time_s", None),
            "slice_time_s": slicing_stats.get("slice_time_s", slicing_stats.get("slicing_time_s", None)),
            "scan_time_s": scanning_stats.get("scan_time_s", scanning_stats.get("scanning_time_s", None)),

            "geometry_generation_and_filter_time_s": geometry_total_time_s,
            "total_run_time_s": total_run_time_s,
            "toolpath_csv": str(toolpath_csv_path),
            "geometry_stl": str(geometry_stl_path),
        }

        summary_row.update(filter_summary)
        summary_row.update(slicing_stats)
        summary_row.update(scanning_stats)

        summary_rows.append(summary_row)
        save_current_logs(summary_rows, filter_attempt_rows, error_rows)

        print(f"Accepted run {run_id}/{N_ACCEPTED_RUNS}")
        print(f"  Slices: {slicing_stats['slice_count']}")
        print(f"  Scan lines: {scanning_stats['scan_line_count']}")
        print(f"  Total scan length: {scanning_stats['total_scan_length_m']:.3f} m")
        print(f"  Estimated laser scan time: {scanning_stats['estimated_laser_scan_time_min']:.2f} min")
        print(f"  Toolpath saved to: {toolpath_csv_path}")

    except Exception as e:
        error_rows.append({
            "process_attempt": process_attempt,
            "stage": "unexpected_error",
            "error_message": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(),
            "time_s_before_error": time.perf_counter() - process_t0,
        })
        save_current_logs(summary_rows, filter_attempt_rows, error_rows)
        print(f"Error in process attempt {process_attempt}: {type(e).__name__}: {e}")
        print("Restarting with a new process attempt.")
        continue


batch_total_time_s = time.perf_counter() - batch_start_time

total_filter_attempts = len(filter_attempt_rows)
total_filter_rejections = sum(1 for r in filter_attempt_rows if not r.get("accepted_by_filter", False))
batch_filter_rejection_percent = (
    100.0 * total_filter_rejections / total_filter_attempts
    if total_filter_attempts > 0
    else 0.0
)

final_report = {
    "accepted_runs": accepted_runs,
    "target_accepted_runs": N_ACCEPTED_RUNS,
    "process_attempts_used": process_attempt,
    "batch_total_time_s": batch_total_time_s,
    "batch_total_time_min": batch_total_time_s / 60.0,
    "total_filter_attempts": total_filter_attempts,
    "total_filter_rejections": total_filter_rejections,
    "batch_filter_rejection_percent": batch_filter_rejection_percent,
}

with open(RUN_DATA_FOLDER / "final_batch_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)

save_current_logs(summary_rows, filter_attempt_rows, error_rows)

print("\n" + "=" * 70)
print("BATCH FINISHED")
print("=" * 70)
print(f"Accepted runs: {accepted_runs}/{N_ACCEPTED_RUNS}")
print(f"Process attempts used: {process_attempt}")
print(f"Total batch time: {batch_total_time_s / 60:.2f} min")
print(f"Total filter rejection percentage: {batch_filter_rejection_percent:.1f}%")

if len(summary_rows) > 0:
    summary_df_temp = pd.DataFrame(summary_rows)

    print("\nTiming summary over accepted runs:")
    print(f"    Average generation time: {summary_df_temp['generation_time_s'].mean():.2f} s")
    print(f"    Average filter time:     {summary_df_temp['filter_time_s'].mean():.2f} s")
    print(f"    Average slice time:      {summary_df_temp['slice_time_s'].mean():.2f} s")
    print(f"    Average scan time:       {summary_df_temp['scan_time_s'].mean():.2f} s")
    print(f"    Average total run time:  {summary_df_temp['total_run_time_s'].mean():.2f} s")

    print("\nTiming summary of last accepted run:")
    last_summary_row = summary_rows[-1]
    print(f"    Generation time: {last_summary_row['generation_time_s']:.2f} s")
    print(f"    Filter time:     {last_summary_row['filter_time_s']:.2f} s")
    print(f"    Slice time:      {last_summary_row['slice_time_s']:.2f} s")
    print(f"    Scan time:       {last_summary_row['scan_time_s']:.2f} s")
    print(f"    Total run time:  {last_summary_row['total_run_time_s']:.2f} s")
else:
    print("\nNo accepted runs, so no timing summary is available.")

print(f"\nSummary CSV: {SUMMARY_CSV}")
print(f"Filter log CSV: {FILTER_LOG_CSV}")
print(f"Error log CSV: {ERROR_LOG_CSV}")
if SAVE_COMBINED_TOOLPATH_CSV:
    print(f"Combined toolpath CSV: {COMBINED_TOOLPATH_CSV}")
